# End-to-End TD-ICU Mortality Prediction Pipeline



### **From MIMIC-IV Raw Data to Clinical Triage with Uncertainty Quantification**

---

### Notebook Overview
This notebook implements a complete, end-to-end pipeline for **Real-Time Intensive Care Unit (ICU) Mortality Prediction**. It is based on the Temporal-Difference (TD) learning methodology proposed by *Frost et al. (2024)*, designed to handle complex, sparse, and irregular clinical event streams.

Beyond standard mortality risk, this notebook integrates **Monte Carlo (MC) Dropout** to provide robust per-patient confidence bounds, enabling realistic clinical triage simulations.

---

### Core Pipeline Stages

#### 1. Data Preprocessing & Validation
* **CSV to Parquet:** Converts raw PhysioNet MIMIC-IV v3.x CSV shards into optimized, schema-typed Parquet files.
* **HDF5 Construction:** Normalizes events, structures time-series data, and handles feature mapping for high-performance I/O.
* **Integrity Checks:** Validates data structures and fixed-padding logic prior to training.

#### 2. Model Architecture (`pyhealth/models`)
* **`CNNLSTMPredictor`:** A hybrid neural network combining CNN feature extraction with LSTM temporal modeling.
* **`TDICUMortalityModel`:** Employs a dual-network setup (Online + Target) trained via TD-learning to map variable-length event streams to a calibrated mortality horizon.

#### 3. Training & Evaluation
* **Distributed Orchestration:** Streams batched records, applying continuous metric tracking (AUROC, AUPRC, Brier Score).
* **Idempotent Saves:** Safely checkpoints the model state and rolls back if execution drops.

#### 4. Uncertainty Estimation & Clinical Triage
* **MC Dropout Inference:** Executes stochastic forward passes to capture model variance.
* **Confidence Bounds:** Generates 95% credible intervals and identifies high/low-confidence predictions.
* **Clinical Triage Simulation:** Automatically stratifies patients into actionable groups based on risk and uncertainty:
  * **Auto-alert:** High confidence, High risk (> 50%)
  * **Senior review:** Low confidence (System asks for human help)
  * **Standard monitoring:** High confidence, Low risk (< 20%)

---

In [ ]:
!pip install -q polars pyarrow h5py
!pip install -q tensordict torchmetrics

In [ ]:
!pip install -q pyhealth

In [ ]:
!pip uninstall -y pyarrow -q
!pip install -q --no-cache-dir pyarrow
!pip install -q --no-cache-dir pyhealth db-dtypes statsmodels torcheval
# RESTART SESSION AFTER DOING THIS -
# Its required due to the version conflicts in colab.
# Proceed to the next cell after restarting session

# <font color='#2196F3'>MIMIC-IV Data Preprocessing</font>




### **CSV to Normalized Parquet Pipeline**

---

The following cells provide the full data-prep pipeline starting from **PhysioNet's MIMIC-IV v3.x** CSV files. This workflow ensures data integrity and high-performance I/O for downstream machine learning tasks.

### Pipeline Overview
The pipeline processes the raw data in two primary stages:
1.  **Type Conversion:** Transforms raw CSV shards into schema-typed **Parquet** files.
2.  **Normalization:** Builds the event, label, and split files that `build_hdf5_from_parquets` expects.

---

### Input Configuration
This is the layout **PhysioNet** distributes for **MIMIC-IV v3.x** (featuring CSV shards per table).

> **Expected Input Layout**
> Ensure your `csv_root` directory follows this structure:

```text
csv_root/
├── hosp/
│   ├── admissions/
│   |   └── sharded CSVs
│   └── patients/
|       └── sharded CSVs
└── icu/
    ├── chartevents/
    |   └── sharded CSVs
    └── icustays/
    |   └── sharded CSVs
    └── inputevents/
        └── sharded CSVs
```

### Output Artifacts
The preprocessing script generates a set of optimized **Parquet** files under `--out-root`. These are structured to be consumed directly by `build_hdf5_from_parquets`:

---

#### Core Metadata
* `stay_demo.parquet`

#### Normalized Events
* `chart_events_norm.parquet`
* `drug_events_norm.parquet`
* `demo_events_norm.parquet`

#### Labels & Data Splits
* `labels.parquet`
* `splits.parquet`
* `labels_split.parquet`
* `stay_lists/{train, val, test}_stays.parquet`

#### Feature Mapping
* `feature_map.parquet`
* `features.txt`

---

In [ ]:
from __future__ import annotations

import glob
import json
import random
import os
from pathlib import Path
import argparse
import math
import time
from copy import deepcopy
from typing import Any, Dict, List, Mapping, Optional, Tuple
from collections import defaultdict
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, IterableDataset, get_worker_info

from tqdm.auto import tqdm

import numpy as np
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
import h5py

from pyhealth.datasets import SampleEHRDataset
from pyhealth.models import BaseModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [43]:
'''
Declare the following Paths -
1) CSV Path - Path where the CSV files from Physionet are residing
2) Out directory - Path where the output of the processed data should reside
3) Checkpoint directory - Directory where the trained models which reside
'''
csv_root = Path("/content/mimic4")
out_root = Path("/content/data/mimic")
checkpoint_dir = Path("/content/checkpoints")
data_root = out_root

## 1) Declare MIMIC-IV Data Schema and Feature Vocabulary


In [28]:
def get_mimic_dtypes() -> Dict[str, Dict[str, Any]]:
    """Return typed schemas for MIMIC-IV v3.x CSV tables.

    These match the fields used by the TD ICU mortality pipeline; unused
    columns are dropped by downstream steps.

    Returns:
        Mapping from table name to ``{column: polars_dtype}``.
    """
    return {
        "admissions": {
            "subject_id": pl.Int64,
            "hadm_id": pl.Int64,
            "admittime": pl.Datetime,
            "dischtime": pl.Datetime,
            "deathtime": pl.Datetime,
            "admission_type": pl.Utf8,
            "discharge_location": pl.Utf8,
            "insurance": pl.Utf8,
            "language": pl.Utf8,
            "marital_status": pl.Utf8,
            "race": pl.Utf8,
            "edregtime": pl.Datetime,
            "edouttime": pl.Datetime,
            "hospital_expire_flag": pl.Int64,
        },
        "patients": {
            "subject_id": pl.Int64,
            "gender": pl.Utf8,
            "anchor_age": pl.Int64,
            "anchor_year": pl.Int64,
            "anchor_year_group": pl.Utf8,
            "dod": pl.Datetime,
        },
        "chartevents": {
            "subject_id": pl.Int64,
            "hadm_id": pl.Int64,
            "stay_id": pl.Int64,
            "caregiver_id": pl.Int64,
            "charttime": pl.Datetime,
            "storetime": pl.Datetime,
            "itemid": pl.Int64,
            "value": pl.Utf8,
            "valuenum": pl.Float64,
            "valueuom": pl.Utf8,
            "warning": pl.Int64,
        },
        "icustays": {
            "subject_id": pl.Int64,
            "hadm_id": pl.Int64,
            "stay_id": pl.Int64,
            "first_careunit": pl.Utf8,
            "last_careunit": pl.Utf8,
            "intime": pl.Datetime,
            "outtime": pl.Datetime,
            "los": pl.Float64,
        },
        "inputevents": {
            "subject_id": pl.Int64,
            "hadm_id": pl.Int64,
            "stay_id": pl.Int64,
            "caregiver_id": pl.Int64,
            "starttime": pl.Datetime,
            "endtime": pl.Datetime,
            "storetime": pl.Datetime,
            "itemid": pl.Int64,
            "amount": pl.Float64,
            "amountuom": pl.Utf8,
            "rate": pl.Float64,
            "rateuom": pl.Utf8,
            "orderid": pl.Int64,
            "linkorderid": pl.Int64,
            "ordercategoryname": pl.Utf8,
            "secondaryordercategoryname": pl.Utf8,
            "ordercomponenttypedescription": pl.Utf8,
            "ordercategorydescription": pl.Utf8,
            "patientweight": pl.Float64,
            "totalamount": pl.Float64,
            "totalamountuom": pl.Utf8,
            "isopenbag": pl.Int64,
            "continueinnextdept": pl.Int64,
            "statusdescription": pl.Utf8,
            "originalamount": pl.Float64,
            "originalrate": pl.Float64,
        },
    }

LAB_ITEMIDS: Dict[int, str] = {
    220045: "Heart Rate",
    220050: "Arterial BP Systolic",
    220051: "Arterial BP Diastolic",
    220052: "Arterial BP Mean",
    220179: "Non-Invasive BP Sys",
    220180: "Non-Invasive BP Dia",
    220181: "Non-Invasive BP Mean",
    220210: "Respiratory Rate",
    220277: "SpO2",
    223761: "Temperature",
    220739: "GCS Eye",
    223900: "GCS Verbal",
    223901: "GCS Motor",
    226253: "Albumin",
    225624: "Bilirubin",
    220635: "Calcium",
    220615: "Creatinine",
    220228: "Haemoglobin",
    220545: "Haematocrit",
    225664: "Bedside Glucose",
    220621: "Glucose",
    227442: "Potassium",
    220734: "pH",
    220235: "Blood Gas pCO2",
    220224: "Blood Gas pO2",
    225690: "Bicarbonate",
    220602: "Chloride",
    220645: "Sodium",
    220507: "Urea",
    220562: "Platelets",
    220546: "WBC",
    227457: "Lactate",
    225668: "Troponin - T",
    220576: "CRP",
    220274: "FiO2",
    227013: "PEEP",
    224685: "Tidal Volume",
    224684: "Minute Volume",
    224695: "Peak Inspiratory Pressure",
    224688: "Respiratory Rate (set)",
    224690: "Respiratory Rate (total)",
    220659: "PTT",
    220613: "ALT",
    220587: "AST",
    226559: "Foley",
}

DRUG_ITEMIDS: Dict[int, str] = {
    221906: "Noradrenaline",
    221289: "Adrenaline",
    221662: "Dopamine",
    221653: "Dobutamine",
    222315: "Vasopressin",
    229581: "Phenylephrine",
    225942: "Propofol",
    221744: "Midazolam",
    222168: "Fentanyl",
    225154: "Morphine Sulfate",
    221749: "Lorazepam",
    222011: "Cisatracurium",
    221986: "Rocuronium",
    225916: "Dexmedetomidine",
    229582: "Ketamine",
    220995: "Heparin",
    228339: "Insulin - Regular",
    222021: "Furosemide",
}

ANTIBIOTIC_ITEMIDS: Dict[int, str] = {
    225798: "Vancomycin",
    225851: "Piperacillin-Tazobactam",
    225893: "Meropenem",
    225879: "Cefepime",
    225875: "Ampicillin",
    225876: "Ampicillin-Sulbactam",
    229584: "Azithromycin",
    229585: "Ceftriaxone",
    225899: "Metronidazole",
    225883: "Levofloxacin",
    228201: "Ciprofloxacin",
    225837: "Fluconazole",
}

# Unit conversions to harmonize across MIMIC-IV's mixed recording units.
LAB_UNIT_CONV: Dict[str, float] = {
    "Albumin": 10.0,
    "Bedside Glucose": 1 / 18,
    "Glucose": 1 / 18,
    "Bilirubin": 17.1,
    "Blood Gas pO2": 0.133322,
    "Blood Gas pCO2": 0.133322,
    "Calcium": 0.2495,
    "Creatinine": 88.42,
    "Haemoglobin": 10.0,
    "Urea": 0.357,
}

LABEL_KEYS: List[str] = [
    "1-day-died", "3-day-died", "7-day-died", "14-day-died", "28-day-died",
]

def build_feature_vocabulary() -> Tuple[List[str], Dict[str, int]]:
    """Construct the ordered feature list and name->id mapping.

    Returns:
        ``(feature_names, name_to_id)`` where ``feature_names`` is a sorted
        list of chart + drug features, followed by demographic features.
    """
    all_drug_map = {**DRUG_ITEMIDS, **ANTIBIOTIC_ITEMIDS}
    lab_features = list(LAB_ITEMIDS.values())
    drug_features = (
        [v + "_rate" for v in DRUG_ITEMIDS.values()]
        + [v + "_rate" for v in ANTIBIOTIC_ITEMIDS.values()]
        + [v + "_bolus" for v in DRUG_ITEMIDS.values()]
        + [v + "_bolus" for v in ANTIBIOTIC_ITEMIDS.values()]
    )
    demo_features = ["age", "gender", "patientweight"]
    feature_names = sorted(set(lab_features + drug_features)) + demo_features
    name_to_id = {f: i for i, f in enumerate(feature_names)}
    _ = all_drug_map  # silence unused
    return feature_names, name_to_id



## 2) Convert the CSV to Parquet and build the Parquet Pipeline Stages



In [29]:
def discover_csv_shards(csv_root: Path) -> Dict[str, List[str]]:
    """Find MIMIC-IV CSV shard files per logical table.

    Args:
        csv_root: Root of the MIMIC-IV CSV tree (contains ``hosp/`` and ``icu/``).

    Returns:
        Mapping from table name to sorted list of CSV paths.
    """
    table_files = {
        "admissions": sorted(
            glob.glob(str(csv_root / "hosp" / "admissions" / "*.csv"))
        ),
        "patients": sorted(
            glob.glob(str(csv_root / "hosp" / "patients" / "*.csv"))
        ),
        "chartevents": sorted(
            glob.glob(str(csv_root / "icu" / "chartevents" / "*.csv"))
        ),
        "icustays": sorted(
            glob.glob(str(csv_root / "icu" / "icustays" / "*.csv"))
        ),
        "inputevents": sorted(
            glob.glob(str(csv_root / "icu" / "inputevents" / "*.csv"))
        ),
    }
    for table, files in table_files.items():
        print(f"  {table}: {len(files)} shard(s)")
    missing = [t for t, fs in table_files.items() if len(fs) == 0]
    if missing:
        raise FileNotFoundError(
            f"No CSV shards found for {missing} under {csv_root}. "
            "Check that the MIMIC-IV download is extracted with the "
            "expected layout (hosp/... and icu/...)."
        )
    return table_files

def csv_to_parquet(
    csv_root: Path,
    parquet_dir: Path,
    overwrite: bool = False,
) -> None:
    """Convert sharded MIMIC-IV CSVs to typed parquet files.

    Uses polars ``scan_csv`` + ``sink_parquet`` so the conversion
    streams and does not require the full CSV in memory.

    Args:
        csv_root: Root of the MIMIC-IV CSV tree.
        parquet_dir: Output directory for per-table parquet files.
        overwrite: If ``False``, skip tables whose parquet already exists.
    """
    parquet_dir.mkdir(parents=True, exist_ok=True)
    table_files = discover_csv_shards(csv_root)
    dtypes = get_mimic_dtypes()

    for table, files in table_files.items():
        out_path = parquet_dir / f"{table}.parquet"
        if out_path.exists() and not overwrite:
            print(f"  skipping {table}: {out_path} exists")
            continue
        print(f"  converting {table}: {len(files)} shard(s) -> {out_path}")
        (
            pl.scan_csv(
                files,
                schema_overrides=dtypes[table],
                infer_schema_length=10000,
                null_values=["", "NULL", "null", "NaN", "nan"],
                ignore_errors=False,
            )
            .sink_parquet(str(out_path))
        )

def build_stay_demographics(
    parquet_dir: Path,
    out_dir: Path,
) -> Path:
    """Join admissions, patients, and icustays into a stay-level table.

    Args:
        parquet_dir: Directory with per-table parquet files.
        out_dir: Destination for ``stay_demo.parquet``.

    Returns:
        Path to the written parquet.
    """
    admissions = pl.scan_parquet(parquet_dir / "admissions.parquet")
    patients = pl.scan_parquet(parquet_dir / "patients.parquet")
    icustays = pl.scan_parquet(parquet_dir / "icustays.parquet")

    stay_demo = (
        icustays
        .join(
            admissions.select([
                "subject_id", "hadm_id", "admittime", "dischtime",
                "deathtime", "hospital_expire_flag",
            ]),
            on=["subject_id", "hadm_id"], how="left",
        )
        .join(
            patients.select(["subject_id", "gender", "anchor_age", "dod"]),
            on="subject_id", how="left",
        )
        .with_columns([
            pl.col("gender").replace({"F": 1.0, "M": 0.0})
              .cast(pl.Float64).alias("gender_code"),
            pl.col("anchor_age").cast(pl.Float64).alias("age"),
        ])
    )

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "stay_demo.parquet"
    stay_demo.collect(engine="streaming").write_parquet(out_path)
    print(f"  wrote {out_path}")
    return out_path

def build_chart_events(
    parquet_dir: Path,
    out_dir: Path,
) -> Path:
    """Build normalized chart events (labs + vitals) with unit harmonization.

    Filters to the itemids in ``LAB_ITEMIDS``, applies
    ``LAB_UNIT_CONV`` where required, and produces rows keyed by
    ``(subject_id, hadm_id, stay_id, charttime, feature, value_num)``.

    Args:
        parquet_dir: Directory with per-table parquet files.
        out_dir: Destination for ``chart_events_norm.parquet``.

    Returns:
        Path to the written parquet.
    """
    chartevents = pl.scan_parquet(parquet_dir / "chartevents.parquet")
    lab_ids = list(LAB_ITEMIDS.keys())

    chart_events = (
        chartevents
        .filter(pl.col("itemid").is_in(lab_ids))
        .filter(pl.col("valuenum").is_not_null())
        .with_columns([
            pl.col("itemid").replace_strict(LAB_ITEMIDS).alias("feature"),
            pl.col("valuenum").cast(pl.Float64).alias("value_num"),
        ])
        .with_columns([
            pl.when(pl.col("feature").is_in(list(LAB_UNIT_CONV.keys())))
              .then(
                  pl.col("value_num")
                  * pl.col("feature").replace_strict(
                      LAB_UNIT_CONV, default=1.0
                  )
              )
              .otherwise(pl.col("value_num"))
              .alias("value_num"),
        ])
        .select([
            "subject_id", "hadm_id", "stay_id", "charttime",
            "feature", "value_num",
        ])
    )

    out_path = out_dir / "chart_events_norm.parquet"
    chart_events.collect(engine="streaming").write_parquet(out_path)
    print(f"  wrote {out_path}")
    return out_path

def build_drug_events(
    parquet_dir: Path,
    out_dir: Path,
) -> Path:
    """Build normalized drug events (rates + boluses).

    Args:
        parquet_dir: Directory with per-table parquet files.
        out_dir: Destination for ``drug_events_norm.parquet``.

    Returns:
        Path to the written parquet.
    """
    inputevents = pl.scan_parquet(parquet_dir / "inputevents.parquet")
    all_drug_map = {**DRUG_ITEMIDS, **ANTIBIOTIC_ITEMIDS}
    all_drug_ids = list(all_drug_map.keys())

    drug_events = (
        inputevents
        .filter(pl.col("itemid").is_in(all_drug_ids))
        .with_columns([
            pl.col("itemid").replace_strict(all_drug_map).alias("drug_name"),
            pl.col("amount").cast(pl.Float64),
            pl.col("rate").cast(pl.Float64),
        ])
        .with_columns([
            pl.when(pl.col("rate").is_not_null())
              .then(pl.col("drug_name") + pl.lit("_rate"))
              .otherwise(pl.col("drug_name") + pl.lit("_bolus"))
              .alias("feature"),
            pl.when(pl.col("rate").is_not_null())
              .then(pl.col("rate"))
              .otherwise(pl.col("amount"))
              .alias("value_num"),
            pl.col("starttime").alias("charttime"),
        ])
        .filter(pl.col("value_num").is_not_null())
        .select([
            "subject_id", "hadm_id", "stay_id", "charttime",
            "feature", "value_num", "patientweight",
        ])
    )

    out_path = out_dir / "drug_events_norm.parquet"
    drug_events.collect(engine="streaming").write_parquet(out_path)
    print(f"  wrote {out_path}")
    return out_path


def build_labels_and_splits(
    stay_demo_path: Path,
    out_dir: Path,
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    seed: int = 42,
) -> Tuple[Path, Path, Path]:
    """Build stay-level mortality labels and patient-level splits.

    Labels are derived from ``dod - intime`` using the five horizons
    from the paper. Splits are patient-level (no subject appears in
    more than one split).

    Args:
        stay_demo_path: Path to ``stay_demo.parquet``.
        out_dir: Directory for output parquets.
        train_frac: Fraction of patients assigned to train.
        val_frac: Fraction assigned to val (remainder goes to test).
        seed: RNG seed for the patient shuffle.

    Returns:
        Triple ``(labels_path, splits_path, labels_split_path)``.
    """
    stay_demo = pl.scan_parquet(stay_demo_path)

    hour_bounds = {
        "1-day-died": 24,
        "3-day-died": 72,
        "7-day-died": 168,
        "14-day-died": 336,
        "28-day-died": 672,
    }

    labels = (
        stay_demo
        .with_columns([
            (
                (pl.col("dod").is_not_null())
                & (
                    (pl.col("dod") - pl.col("intime"))
                    .dt.total_hours() <= hrs
                )
            )
            .cast(pl.Int8).alias(key)
            for key, hrs in hour_bounds.items()
        ])
        .select(
            ["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
            + list(hour_bounds.keys())
            + ["age", "gender_code"]
        )
    )
    labels_path = out_dir / "labels.parquet"
    labels.collect(engine="streaming").write_parquet(labels_path)
    print(f"  wrote {labels_path}")

    # Patient-level split
    labels_df = pl.read_parquet(labels_path)
    subject_ids = labels_df["subject_id"].unique().to_numpy()
    rng = np.random.default_rng(seed)
    rng.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_ids = set(subject_ids[:n_train].tolist())
    val_ids = set(subject_ids[n_train:n_train + n_val].tolist())

    split_df = labels_df.select(["subject_id", "stay_id"]).with_columns([
        pl.when(pl.col("subject_id").is_in(train_ids)).then(pl.lit("train"))
        .when(pl.col("subject_id").is_in(val_ids)).then(pl.lit("val"))
        .otherwise(pl.lit("test"))
        .alias("split"),
    ])
    splits_path = out_dir / "splits.parquet"
    split_df.write_parquet(splits_path)
    print(f"  wrote {splits_path}")

    labels_split = labels_df.join(
        split_df, on=["subject_id", "stay_id"], how="left",
    )
    labels_split_path = out_dir / "labels_split.parquet"
    labels_split.write_parquet(labels_split_path)
    print(f"  wrote {labels_split_path}")

    return labels_path, splits_path, labels_split_path

def build_demo_events(
    stay_demo_path: Path,
    labels_split_path: Path,
    drug_events_path: Path,
    out_dir: Path,
) -> Path:
    """Build demographic event rows (age, gender, weight) at ICU admission.

    Args:
        stay_demo_path: Path to ``stay_demo.parquet``.
        labels_split_path: Path to ``labels_split.parquet``.
        drug_events_path: Path to ``drug_events_norm.parquet`` (used to
            pull the first-recorded ``patientweight`` per stay).
        out_dir: Destination for ``demo_events_norm.parquet``.

    Returns:
        Path to the written parquet.
    """
    stay_demo = pl.scan_parquet(stay_demo_path).select([
        "subject_id", "hadm_id", "stay_id", "intime", "age", "gender_code",
    ])
    labels_split = pl.scan_parquet(labels_split_path).select([
        "subject_id", "hadm_id", "stay_id", "split",
    ])
    drug_events = pl.scan_parquet(drug_events_path)

    weight_events = (
        drug_events
        .filter(pl.col("patientweight").is_not_null())
        .group_by(["subject_id", "hadm_id", "stay_id"])
        .agg([
            pl.col("patientweight").drop_nulls().first().alias("patientweight"),
        ])
    )

    demo_base = (
        stay_demo
        .join(
            labels_split,
            on=["subject_id", "hadm_id", "stay_id"], how="left",
        )
        .join(
            weight_events,
            on=["subject_id", "hadm_id", "stay_id"], how="left",
        )
        .with_columns([pl.col("intime").alias("charttime")])
        .select([
            "subject_id", "hadm_id", "stay_id", "charttime", "split",
            "age", "gender_code", "patientweight",
        ])
    )

    def _make_event_rows(
        col: str, feature_name: str,
    ) -> pl.LazyFrame:
        return (
            demo_base
            .filter(pl.col(col).is_not_null())
            .select([
                "subject_id", "hadm_id", "stay_id",
                "charttime", "split", col,
            ])
            .with_columns([
                pl.lit(feature_name).alias("feature"),
                pl.col(col).cast(pl.Float64).alias("value_num"),
            ])
            .select([
                "subject_id", "hadm_id", "stay_id", "charttime",
                "split", "feature", "value_num",
            ])
        )

    age_rows = _make_event_rows("age", "age")
    gender_rows = _make_event_rows("gender_code", "gender")
    weight_rows = _make_event_rows("patientweight", "patientweight")

    demo_event_rows = pl.concat([age_rows, gender_rows, weight_rows])
    out_path = out_dir / "demo_events_norm.parquet"
    demo_event_rows.collect(engine="streaming").write_parquet(out_path)
    print(f"  wrote {out_path}")
    return out_path

def build_stay_lists(
    labels_split_path: Path,
    out_dir: Path,
) -> Dict[str, Path]:
    """Write per-split stay-id lists to ``{out_dir}/stay_lists/``.

    Args:
        labels_split_path: Path to ``labels_split.parquet``.
        out_dir: Directory that will contain ``stay_lists/``.

    Returns:
        Mapping from split name to parquet path.
    """
    stay_list_dir = out_dir / "stay_lists"
    stay_list_dir.mkdir(parents=True, exist_ok=True)
    labels_split = pl.read_parquet(labels_split_path)

    out: Dict[str, Path] = {}
    for split_name in ["train", "val", "test"]:
        p = stay_list_dir / f"{split_name}_stays.parquet"
        (
            labels_split
            .filter(pl.col("split") == split_name)
            .select(["subject_id", "hadm_id", "stay_id"])
            .unique()
            .write_parquet(p)
        )
        print(f"  wrote {p}")
        out[split_name] = p
    return out

def build_feature_map(out_dir: Path) -> Tuple[Path, Path]:
    """Write ``feature_map.parquet`` and ``features.txt``.

    Args:
        out_dir: Destination directory.

    Returns:
        Pair of paths ``(feature_map_path, features_txt_path)``.
    """
    feature_names, name_to_id = build_feature_vocabulary()
    feat_df = pl.DataFrame({
        "feature": list(name_to_id.keys()),
        "feature_id": list(name_to_id.values()),
    })
    feature_map_path = out_dir / "feature_map.parquet"
    feat_df.write_parquet(feature_map_path)
    print(f"  wrote {feature_map_path}")

    features_txt = out_dir / "features.txt"
    features_txt.write_text("\n".join(feature_names) + "\n")
    print(f"  wrote {features_txt}")

    return feature_map_path, features_txt

## 3) Pipeline Orchestrator
### Run the above pipeline



---

### 🚨 Prerequisites
> **CRITICAL:** Ensure all raw CSV files are correctly loaded into your `csv_root` directory before continuing. The orchestrator will fail if it cannot locate the `hosp/` and `icu/` sub-directories.

---



In [ ]:
out_root.mkdir(parents=True, exist_ok=True)

parquet_dir = out_root / "tables"
overwrite_parquets = False

print("\n[1/7] Converting CSV -> typed parquet...")
csv_to_parquet(csv_root, parquet_dir, overwrite=overwrite_parquets)

print("\n[2/7] Building stay_demo...")
stay_demo_path = build_stay_demographics(parquet_dir, out_root)

print("\n[3/7] Building chart events (labs + vitals)...")
build_chart_events(parquet_dir, out_root)

print("\n[4/7] Building drug events...")
drug_events_path = build_drug_events(parquet_dir, out_root)

print("\n[5/7] Building labels + splits...")
_, _, labels_split_path = build_labels_and_splits(
        stay_demo_path, out_root,
    )

print("\n[6/7] Building demographic event rows + stay lists...")
build_demo_events(
        stay_demo_path, labels_split_path, drug_events_path, out_root,
    )
build_stay_lists(labels_split_path, out_root)

print("\n[7/7] Writing feature map + features.txt...")
build_feature_map(out_root)

print(f"\nAll preprocessing outputs live under {out_root}")

## 4) Compute Scaling

In [31]:
def compute_scaling_from_h5(
    h5_path: Path,
    feature_names: List[str],
    sample_rows: Optional[int] = None,
) -> Dict[str, Any]:
    """Compute per-feature mean and std from a training HDF5.

    Sweeps the training split once with running sums. For each of
    ``values``, ``delta_time`` and ``delta_value``, computes per-feature
    stats using only non-NaN entries and feature-id >= 0 slots.

    Args:
        h5_path: Path to the training HDF5 file.
        feature_names: Ordered feature names (length == n_features).
        sample_rows: If provided, only use the first ``sample_rows`` rows.

    Returns:
        Scaling dict in the format expected by ``TDICUMortalityModel``.
    """
    import h5py

    n_features = len(feature_names)
    value_sum = np.zeros(n_features, dtype=np.float64)
    value_sq = np.zeros(n_features, dtype=np.float64)
    value_cnt = np.zeros(n_features, dtype=np.int64)
    dt_sum = np.zeros(n_features, dtype=np.float64)
    dt_sq = np.zeros(n_features, dtype=np.float64)
    dt_cnt = np.zeros(n_features, dtype=np.int64)
    dv_sum = np.zeros(n_features, dtype=np.float64)
    dv_sq = np.zeros(n_features, dtype=np.float64)
    dv_cnt = np.zeros(n_features, dtype=np.int64)
    tp_sum, tp_sq, tp_cnt = 0.0, 0.0, 0

    with h5py.File(h5_path, "r") as h5f:
        n = h5f["features"].shape[0]
        rows = n if sample_rows is None else min(sample_rows, n)
        chunk = 2048
        for i in range(0, rows, chunk):
            j = min(i + chunk, rows)
            feats = h5f["features"][i:j]
            vals = h5f["values"][i:j]
            dts = h5f["deltatime"][i:j]
            dvs = h5f["deltavalue"][i:j]
            tps = h5f["timepoints"][i:j]

            feat_valid = feats >= 0
            tp_valid = ~np.isnan(tps)
            tp = tps[tp_valid]
            tp_sum += tp.sum()
            tp_sq += np.square(tp).sum()
            tp_cnt += tp.size

            for arr, sum_, sq_, cnt_ in [
                (vals, value_sum, value_sq, value_cnt),
                (dts, dt_sum, dt_sq, dt_cnt),
                (dvs, dv_sum, dv_sq, dv_cnt),
            ]:
                valid = feat_valid & ~np.isnan(arr)
                idx = feats[valid].astype(np.int64)
                x = arr[valid].astype(np.float64)
                np.add.at(sum_, idx, x)
                np.add.at(sq_, idx, x * x)
                np.add.at(cnt_, idx, 1)

    def finalize(sum_, sq_, cnt_) -> Tuple[np.ndarray, np.ndarray]:
        """Convert running sums to (mean, std) with a small-variance floor."""
        mean = np.divide(sum_, np.maximum(cnt_, 1))
        var = np.divide(sq_, np.maximum(cnt_, 1)) - mean ** 2
        var = np.maximum(var, 1e-8)
        std = np.sqrt(var)
        return mean.astype(np.float32), std.astype(np.float32)

    v_mean, v_std = finalize(value_sum, value_sq, value_cnt)
    dt_mean, dt_std = finalize(dt_sum, dt_sq, dt_cnt)
    dv_mean, dv_std = finalize(dv_sum, dv_sq, dv_cnt)
    tp_mean = np.float32(tp_sum / max(tp_cnt, 1))
    tp_var = max(tp_sq / max(tp_cnt, 1) - tp_mean ** 2, 1e-8)
    tp_std = np.float32(np.sqrt(tp_var))

    def _per_feature(arr: np.ndarray) -> Dict[str, torch.Tensor]:
        """Expand a 1D per-feature array into the name-keyed tensor dict."""
        return {
            f: torch.tensor([arr[i]], dtype=torch.float32)
            for i, f in enumerate(feature_names)
        }

    return {
        "mean": {
            "timepoints": torch.tensor([tp_mean], dtype=torch.float32),
            "values": _per_feature(v_mean),
            "delta_time": _per_feature(dt_mean),
            "delta_value": _per_feature(dv_mean),
        },
        "std": {
            "timepoints": torch.tensor([tp_std], dtype=torch.float32),
            "values": _per_feature(v_std),
            "delta_time": _per_feature(dt_std),
            "delta_value": _per_feature(dv_std),
        },
    }


##5) This stage converts the parquet -> HDF5


In [32]:

NEXT_STATE_DELAY_HOURS = 24.0
NEXT_STATE_WINDOW_HOURS = 24.0


def choose_sample_indices(
    n_events: int,
    first_k: int = 8,
    every_k: int = 32,
    max_samples: int = 100,
) -> List[int]:
    """Pick state-marker indices within a single ICU stay.

    Args:
        n_events: Number of events in the stay.
        first_k: Keep every event in the first k.
        every_k: After the first k, sample every k'th event.
        max_samples: Cap on total samples per stay.

    Returns:
        Sorted list of indices into the stay's event array.
    """
    if n_events <= 1:
        return []
    idx_set = set()
    for i in range(min(first_k, n_events - 1)):
        idx_set.add(i)
    i = first_k
    while i < n_events - 1:
        idx_set.add(i)
        i += every_k
    idx_set.add(n_events - 1)
    idx_list = sorted(idx_set)

    if len(idx_list) > max_samples:
        first_block = idx_list[:first_k]
        terminal = idx_list[-1]
        middle = idx_list[first_k:-1]
        budget = max_samples - first_k - 1
        if budget > 0 and len(middle) > budget:
            step = len(middle) / budget
            middle = [middle[int(i * step)] for i in range(budget)]
        idx_list = first_block + middle + [terminal]
    return idx_list


def get_next_state_idxs_batch(
    times: np.ndarray,
    cur_idxs: np.ndarray,
    delay: float = NEXT_STATE_DELAY_HOURS,
    window: float = NEXT_STATE_WINDOW_HOURS,
) -> Tuple[np.ndarray, np.ndarray]:
    """Vectorized next-state selection per the paper's SMRP rule.

    Args:
        times: Sorted 1D float array of event times for one stay.
        cur_idxs: Integer array of current-state indices.
        delay: SMRP delay in hours.
        window: Eligibility window length in hours.

    Returns:
        Tuple ``(next_idxs, is_terminal)``.
    """
    cur_idxs = np.asarray(cur_idxs, dtype=np.int64)
    t0 = times[cur_idxs]
    lo = t0 + delay
    hi = t0 + delay + window
    next_idxs = np.searchsorted(times, lo, side="left")
    in_bounds = next_idxs < len(times)
    safe_next = np.clip(next_idxs, 0, len(times) - 1)
    within = times[safe_next] <= hi
    after = next_idxs > cur_idxs
    valid = in_bounds & within & after
    is_terminal = ~valid
    next_idxs = np.where(is_terminal, cur_idxs, next_idxs).astype(np.int64)
    return next_idxs, is_terminal


def precompute_dt(times: np.ndarray) -> np.ndarray:
    """Compute gap-from-previous-event for a whole stay.

    Args:
        times: Sorted timepoints.

    Returns:
        Same-length dt array with ``dt[0] = 0``.
    """
    n = len(times)
    dt = np.zeros(n, dtype=np.float32)
    if n > 1:
        dt[1:] = np.diff(times)
    return dt


def precompute_dv(values: np.ndarray, features: np.ndarray) -> np.ndarray:
    """Compute value-change-since-previous-measurement for a whole stay.

    Args:
        values: Event values.
        features: Event feature ids.

    Returns:
        Same-length dv array.
    """
    n = len(values)
    dv = np.zeros(n, dtype=np.float32)
    if n == 0:
        return dv
    idx = np.argsort(features, kind="stable")
    f_sorted = features[idx]
    v_sorted = values[idx]
    is_new = np.empty(n, dtype=bool)
    is_new[0] = True
    is_new[1:] = f_sorted[1:] != f_sorted[:-1]
    dv_sorted = np.empty(n, dtype=np.float32)
    dv_sorted[0] = 0.0
    dv_sorted[1:] = v_sorted[1:] - v_sorted[:-1]
    dv_sorted[is_new] = 0.0
    dv[idx] = dv_sorted
    return dv


def build_window_arrays(
    times: np.ndarray,
    values: np.ndarray,
    features: np.ndarray,
    dt_full: np.ndarray,
    dv_full: np.ndarray,
    end_idx: int,
    context_len: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Build a single padded state window ending at ``end_idx``.

    Args:
        times: Full-stay timepoints.
        values: Full-stay values.
        features: Full-stay feature ids.
        dt_full: Pre-computed full-stay dt array.
        dv_full: Pre-computed full-stay dv array.
        end_idx: Terminal event index.
        context_len: Target window size.

    Returns:
        Tuple of arrays ``(timepoints, values, features, deltatime, deltavalue)``
        each of length ``context_len``. Head slots are NaN / -1 padded.
    """
    start_idx = max(0, end_idx - context_len + 1)
    t = times[start_idx:end_idx + 1]
    v = values[start_idx:end_idx + 1]
    f = features[start_idx:end_idx + 1]
    dt = dt_full[start_idx:end_idx + 1].copy()
    dv = dv_full[start_idx:end_idx + 1]
    if len(dt) > 0:
        dt[0] = 0.0
    pad_n = context_len - len(t)

    timepoints = np.full((context_len,), np.nan, dtype=np.float32)
    values_arr = np.full((context_len,), np.nan, dtype=np.float32)
    features_arr = np.full((context_len,), -1, dtype=np.int16)
    deltatime = np.full((context_len,), -1.0, dtype=np.float32)
    deltavalue = np.full((context_len,), np.nan, dtype=np.float32)

    timepoints[pad_n:] = t
    values_arr[pad_n:] = v
    features_arr[pad_n:] = f
    deltatime[pad_n:] = dt
    deltavalue[pad_n:] = dv
    return timepoints, values_arr, features_arr, deltatime, deltavalue


In [33]:
def build_hdf5_from_parquets(
    parquet_dir: Path,
    out_dir: Path,
    split: str,
    context_len: int = 400,
    first_k: int = 8,
    every_k: int = 32,
    max_samples_per_stay: int = 100,
    label_keys: Optional[List[str]] = None,
    flush_every: int = 8192,
    stay_batch_size: int = 2000,
) -> Path:
    """Build one split's HDF5 from normalized MIMIC-IV parquet files.

    Expected files under ``parquet_dir``:

    * ``chart_events_norm.parquet``  with columns ``subject_id``, ``hadm_id``,
      ``stay_id``, ``charttime``, ``feature``, ``value_num``.
    * ``drug_events_norm.parquet``   same schema.
    * ``demo_events_norm.parquet``   same schema.
    * ``labels_split.parquet``       with ``subject_id``, ``hadm_id``,
      ``stay_id``, ``intime``, ``outtime``, ``split``, plus binary label
      columns (``1-day-died`` ... ``28-day-died``).
    * ``stay_lists/{split}_stays.parquet`` with ``stay_id`` column.
    * ``feature_map.parquet`` with ``feature`` (str) and ``feature_id`` (int).

    Output: ``{out_dir}/h5{split}_fixed.hdf5``.

    Args:
        parquet_dir: Directory containing the parquet inputs.
        out_dir: Destination directory for the HDF5 file.
        split: One of ``"train"``, ``"val"``, ``"test"``.
        context_len: Sequence length per state window.
        first_k: ``choose_sample_indices`` parameter.
        every_k: ``choose_sample_indices`` parameter.
        max_samples_per_stay: Cap on samples per ICU stay.
        label_keys: Binary label columns in ``labels_split.parquet``.
            Defaults to the 5 horizons from the paper.
        flush_every: Rows to buffer between HDF5 writes.
        stay_batch_size: Stays per polars batch load.

    Returns:
        Path to the written HDF5 file.
    """
    import h5py
    import polars as pl

    if label_keys is None:
        label_keys = [
            "1-day-died", "3-day-died", "7-day-died",
            "14-day-died", "28-day-died",
        ]

    chart_path = parquet_dir / "chart_events_norm.parquet"
    drug_path = parquet_dir / "drug_events_norm.parquet"
    demo_path = parquet_dir / "demo_events_norm.parquet"
    labels_path = parquet_dir / "labels_split.parquet"
    stay_list_path = parquet_dir / "stay_lists" / f"{split}_stays.parquet"
    feature_map_path = parquet_dir / "feature_map.parquet"

    if not feature_map_path.exists():
        raise FileNotFoundError(
            f"feature_map.parquet not found at {feature_map_path}. "
            "Build this from the training split before running build mode."
        )
    feat_df = pl.read_parquet(feature_map_path)

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"h5{split}_fixed.hdf5"
    if out_path.exists():
        out_path.unlink()

    h5f = h5py.File(out_path, "w", libver="latest")
    c = context_len
    chunk_2d = (4096, c)
    chunk_1d = (min(4096, 65536),)
    kw: Dict[str, Any] = dict(compression=None, track_times=False)

    def _create_2d(name: str, dtype: str) -> Any:
        """Create a resizable 2D HDF5 dataset of shape (0, context_len)."""
        return h5f.create_dataset(
            name, shape=(0, c), maxshape=(None, c),
            chunks=chunk_2d, dtype=dtype, **kw,
        )

    def _create_1d(name: str, dtype: str) -> Any:
        """Create a resizable 1D HDF5 dataset of shape (0,)."""
        return h5f.create_dataset(
            name, shape=(0,), maxshape=(None,),
            chunks=chunk_1d, dtype=dtype, **kw,
        )

    datasets: Dict[str, Any] = {
        "timepoints": _create_2d("timepoints", "f4"),
        "values": _create_2d("values", "f4"),
        "features": _create_2d("features", "i2"),
        "deltatime": _create_2d("deltatime", "f4"),
        "deltavalue": _create_2d("deltavalue", "f4"),
        "nexttimepoints": _create_2d("nexttimepoints", "f4"),
        "nextvalues": _create_2d("nextvalues", "f4"),
        "nextfeatures": _create_2d("nextfeatures", "i2"),
        "nextdeltatime": _create_2d("nextdeltatime", "f4"),
        "nextdeltavalue": _create_2d("nextdeltavalue", "f4"),
        "isterminal": _create_1d("isterminal", "i1"),
    }
    for key in label_keys:
        datasets[key] = _create_1d(key, "i1")

    stay_ids = (
        pl.read_parquet(stay_list_path)
        .select(["stay_id"]).unique().sort("stay_id")["stay_id"].to_list()
    )

    def _flush(store: Dict[str, List]) -> None:
        """Append buffered rows to HDF5 datasets, then clear the buffer."""
        n = len(store["isterminal"])
        if n == 0:
            return
        for k, arr_list in store.items():
            if k in label_keys or k == "isterminal":
                arr = np.asarray(arr_list, dtype=np.int8)
            elif k in ("features", "nextfeatures"):
                arr = np.stack(arr_list).astype(np.int16, copy=False)
            else:
                arr = np.stack(arr_list).astype(np.float32, copy=False)
            ds = datasets[k]
            old_n = ds.shape[0]
            new_n = old_n + len(arr)
            if arr.ndim == 1:
                ds.resize((new_n,))
                ds[old_n:new_n] = arr
            else:
                ds.resize((new_n, arr.shape[1]))
                ds[old_n:new_n, :] = arr
        store.clear()

    batch_store: Dict[str, List] = defaultdict(list)
    total_written = 0
    n_batches = math.ceil(len(stay_ids) / stay_batch_size)
    t0 = time.time()

    for b in range(n_batches):
        lo = b * stay_batch_size
        hi = min((b + 1) * stay_batch_size, len(stay_ids))
        batch_ids = stay_ids[lo:hi]

        split_labels = (
            pl.scan_parquet(labels_path)
            .filter(
                (pl.col("split") == split)
                & (pl.col("stay_id").is_in(batch_ids))
            )
            .select(
                ["subject_id", "hadm_id", "stay_id", "intime", *label_keys]
            )
        )
        events = pl.concat([
            pl.scan_parquet(p).filter(pl.col("stay_id").is_in(batch_ids))
            .select([
                "subject_id", "hadm_id", "stay_id", "charttime",
                "feature", "value_num",
            ])
            for p in [chart_path, drug_path, demo_path]
        ])
        events_df = (
            events
            .join(
                split_labels,
                on=["subject_id", "hadm_id", "stay_id"],
                how="inner",
            )
            .join(feat_df.lazy(), on="feature", how="inner")
            .with_columns([pl.col("feature_id").cast(pl.Int16)])
            .sort(["stay_id", "charttime"])
            .collect(engine="streaming")
        )

        for _stay_key, sdf in events_df.group_by(
            "stay_id", maintain_order=True,
        ):
            sdf = sdf.sort("charttime")
            intime = sdf["intime"][0]
            times = (
                (sdf["charttime"] - intime).dt.total_seconds().to_numpy()
                / 3600.0
            ).astype(np.float32)
            values = sdf["value_num"].to_numpy().astype(np.float32)
            features = sdf["feature_id"].to_numpy().astype(np.int16)
            labels = {k: int(sdf[k][0]) for k in label_keys}

            n_events = len(times)
            sample_idxs = choose_sample_indices(
                n_events, first_k=first_k, every_k=every_k,
                max_samples=max_samples_per_stay,
            )
            if not sample_idxs:
                continue

            dt_full = precompute_dt(times)
            dv_full = precompute_dv(values, features)
            sample_arr = np.asarray(sample_idxs, dtype=np.int64)
            next_arr, term_arr = get_next_state_idxs_batch(times, sample_arr)

            for k, idx in enumerate(sample_idxs):
                cur = build_window_arrays(
                    times, values, features, dt_full, dv_full,
                    idx, context_len,
                )
                if term_arr[k]:
                    nxt = cur
                    is_terminal = 1
                else:
                    nxt = build_window_arrays(
                        times, values, features, dt_full, dv_full,
                        int(next_arr[k]), context_len,
                    )
                    is_terminal = 0

                batch_store["timepoints"].append(cur[0])
                batch_store["values"].append(cur[1])
                batch_store["features"].append(cur[2])
                batch_store["deltatime"].append(cur[3])
                batch_store["deltavalue"].append(cur[4])
                batch_store["nexttimepoints"].append(nxt[0])
                batch_store["nextvalues"].append(nxt[1])
                batch_store["nextfeatures"].append(nxt[2])
                batch_store["nextdeltatime"].append(nxt[3])
                batch_store["nextdeltavalue"].append(nxt[4])
                batch_store["isterminal"].append(is_terminal)
                for lk in label_keys:
                    batch_store[lk].append(labels[lk])

                if len(batch_store["isterminal"]) >= flush_every:
                    _flush(batch_store)
                    total_written = datasets["isterminal"].shape[0]

        print(
            f"  [{split}] batch {b + 1}/{n_batches} | rows={total_written} "
            f"| {(time.time() - t0) / 60:.1f} min"
        )

    _flush(batch_store)
    h5f.close()
    print(f"  [{split}] wrote {out_path}")
    return out_path



## 6) HDF5 -> DataLoader

In [34]:

class H5SequenceDataset(Dataset):
    """Dataset that serves state transitions from a pre-built HDF5 file.

    Opens the file lazily inside each worker process. Expects the HDF5
    layout produced by ``build_hdf5_from_parquets``.

    Args:
        h5_path: Path to the HDF5 file.
        label_keys: Binary label columns stored in the file.
        label_key: The single label column to expose as the primary target.
    """

    def __init__(
        self,
        h5_path: Path,
        label_keys: List[str],
        label_key: str = "28-day-died",
    ) -> None:
        import h5py

        self.h5_path = str(h5_path)
        self.label_keys = label_keys
        self.label_key = label_key
        self._h5 = None
        with h5py.File(self.h5_path, "r") as f:
            self._length = f["features"].shape[0]

    def _open(self) -> None:
        """Open the HDF5 file handle (once per worker process)."""
        if self._h5 is None:
            import h5py

            self._h5 = h5py.File(self.h5_path, "r")

    def __len__(self) -> int:
        return self._length

    def __getitem__(
        self, idx: int,
    ) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor], Dict[str, Any]]:
        """Return one (batch dict, targets dict, meta dict) tuple.

        Args:
            idx: Sample index.

        Returns:
            Tuple of dicts that ``collate_td`` stacks into a batch.
        """
        self._open()
        f = self._h5
        x = {
            "timepoints": torch.from_numpy(f["timepoints"][idx]).float(),
            "values": torch.from_numpy(f["values"][idx]).float(),
            "features": torch.from_numpy(f["features"][idx]).long(),
            "delta_time": torch.from_numpy(f["deltatime"][idx]).float(),
            "delta_value": torch.from_numpy(f["deltavalue"][idx]).float(),
            "next_timepoints": torch.from_numpy(
                f["nexttimepoints"][idx]
            ).float(),
            "next_values": torch.from_numpy(f["nextvalues"][idx]).float(),
            "next_features": torch.from_numpy(f["nextfeatures"][idx]).long(),
            "next_delta_time": torch.from_numpy(
                f["nextdeltatime"][idx]
            ).float(),
            "next_delta_value": torch.from_numpy(
                f["nextdeltavalue"][idx]
            ).float(),
            "isterminal": torch.tensor(
                [f["isterminal"][idx]], dtype=torch.float32
            ),
        }
        y = {
            k: torch.tensor([f[k][idx]], dtype=torch.float32)
            for k in self.label_keys
        }
        meta = {"sample_id": int(idx)}
        return x, y, meta


def collate_td(batch):
    """Stack a list of samples into a batched dict.

    Args:
        batch: List of ``(x_dict, y_dict, meta_dict)`` tuples.

    Returns:
        Triple of stacked dicts plus the list of meta dicts.
    """
    xs, ys, metas = zip(*batch)
    out_x = {k: torch.stack([x[k] for x in xs], dim=0) for k in xs[0]}
    out_y = {k: torch.stack([y[k] for y in ys], dim=0) for k in ys[0]}
    return out_x, out_y, metas


# Model Architecture: TD-ICU Mortality
### **PyHealth Implementation & Uncertainty Extension**

---

This module implements the methodology from:
> **Frost et al.**, *"Robust Real-Time Mortality Prediction in the Intensive Care Unit using Temporal Difference Learning"* ([arXiv:2411.04285](https://arxiv.org/abs/2411.04285)).

Framing mortality prediction as a **value-estimation problem**. Instead of simple classification, the model predicts the probability of death within a fixed horizon (e.g., 28 days) using **Temporal-Difference (TD) targets** derived from a lagged "target network."

---

### Core Components

The implementation provides two specialized classes:

#### 1. `CNNLSTMPredictor`
* **Architecture:** Hybrid CNN + LSTM backbone.
* **Function:** Maps complex event streams to a single mortality probability.
* **Usage:** Ideal for standard supervised training baselines.

#### 2. `TDICUMortalityModel`
* **Framework:** A `PyHealth.BaseModel` wrapper.
* **Logic:** Manages dual networks (**Online** + **Target**) and implements the TD training rule.
* **Inference Plus:** Features `predict_with_confidence`, which uses **Monte Carlo (MC) Dropout** to attach a per-patient uncertainty estimate to every prediction.

---

The cell below is the code from pyhealth/models


In [35]:
class MaxPool1D(nn.Module):
    """NaN-aware 1D max pool for irregular event streams.

    Windows that are entirely NaN remain NaN in the output. Windows that
    contain at least one real value output the max of those real values.
    Padded positions can therefore be tracked across pooling layers.

    Args:
        kernel_size: Pooling window size.
        stride: Pooling stride.
    """

    def __init__(self, kernel_size: int = 2, stride: int = 2) -> None:
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run NaN-aware max pooling.

        Args:
            x: Input tensor of shape ``[batch, channels, seq_len]``.

        Returns:
            Pooled tensor with NaN preserved in all-NaN windows.
        """
        neg_inf = torch.tensor(-np.inf, dtype=x.dtype, device=x.device)
        pos_nan = torch.tensor(np.nan, dtype=x.dtype, device=x.device)
        out = torch.where(torch.isnan(x), neg_inf, x)
        out = F.max_pool1d(out, kernel_size=self.kernel_size, stride=self.stride)
        return torch.where(torch.isinf(out), pos_nan, out)


class Transpose(nn.Module):
    """Swap two tensor dimensions inside an ``nn.Sequential`` pipeline.

    Args:
        dim1: First dimension to swap.
        dim2: Second dimension to swap.
    """

    def __init__(self, dim1: int, dim2: int) -> None:
        super().__init__()
        self.dim1 = dim1
        self.dim2 = dim2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Transpose the configured dimensions.

        Args:
            x: Input tensor.

        Returns:
            Tensor with ``dim1`` and ``dim2`` swapped.
        """
        return x.transpose(self.dim1, self.dim2)


# -----------------------------------------------------------------------------
# CNN + LSTM backbone
# -----------------------------------------------------------------------------


class CNNLSTMPredictor(nn.Module):
    """CNN + LSTM encoder producing mortality predictions.

    The predictor embeds each component of an irregular event stream
    (timepoint, value, feature id, delta-time, delta-value), fuses them via
    summation + batchnorm, applies a small CNN stack for sequence-length
    reduction, a 2-layer LSTM for temporal modelling, and a dense head for
    binary classification.

    Args:
        n_features: Size of the feature vocabulary.
        features: Ordered list of feature names used to assemble the
            scaling buffers from ``scaling``.
        output_dim: Output dimensionality (always 1 for binary mortality).
        scaling: Dictionary of per-feature statistics with the structure
            ``{"mean": {...}, "std": {...}}`` where each inner dict contains
            per-feature 1-element tensors for ``values``, ``delta_time``,
            and ``delta_value``, plus scalar tensors for ``timepoints``.
        cnn_layers: Number of CNN blocks (each applies a conv + ReLU + pool).
        hidden_dim: Channel/embedding dimension of the encoder.
        dropout: Dropout applied between LSTM layers.
        batch_first: Whether the LSTM consumes batch-first tensors.
        dtype: Floating point dtype for model parameters.
        device: Device string, used to choose between packed vs masked LSTM.

    Attributes:
        embedding_net: ``nn.ModuleDict`` of five per-component embeddings.
        cnn: CNN stack that reduces sequence length.
        lstm: Two-layer LSTM with hidden size ``hidden_dim * 8``.
        dense: Binary prediction head.
    """

    def __init__(
        self,
        n_features: int,
        features: List[str],
        output_dim: int,
        scaling: Mapping[str, Any],
        cnn_layers: int = 2,
        hidden_dim: int = 32,
        dropout: float = 0.5,
        batch_first: bool = True,
        dtype: torch.dtype = torch.float32,
        device: str = "cpu",
    ) -> None:
        super().__init__()
        self.dtype = dtype
        self.device_name = device
        self.n_features = n_features
        self.features = features
        self.output_dim = output_dim
        self.cnn_layers = cnn_layers
        self.hidden_dim = hidden_dim

        self._register_scaling_buffers(scaling)
        self._build_embeddings()
        self._build_cnn()
        self._build_lstm(dropout=dropout, batch_first=batch_first)
        self._build_head()

        self.to(device)
        self.init_weights()

    # -- construction helpers -------------------------------------------------

    def _register_scaling_buffers(
        self,
        scaling: Mapping[str, Any],
    ) -> None:
        """Register per-feature mean/std tensors as buffers."""
        for name in ["values", "delta_time", "delta_value"]:
            mean_t = torch.cat([scaling["mean"][name][f] for f in self.features])
            std_t = torch.cat([scaling["std"][name][f] for f in self.features])
            self.register_buffer(f"mean_{name}", mean_t)
            self.register_buffer(f"std_{name}", std_t)
        self.register_buffer("mean_timepoints", scaling["mean"]["timepoints"])
        self.register_buffer("std_timepoints", scaling["std"]["timepoints"])

    def _build_embeddings(self) -> None:
        """Build per-component embedding networks."""
        interim = int(np.sqrt(self.hidden_dim))
        self.embedding_net = nn.ModuleDict()
        for name in ["time", "value", "feature", "delta_time", "delta_value"]:
            if name == "feature":
                self.embedding_net[name] = nn.Embedding(
                    self.n_features, self.hidden_dim, dtype=self.dtype
                )
            else:
                self.embedding_net[name] = nn.Sequential(
                    nn.Linear(1, interim, dtype=self.dtype),
                    nn.ReLU(),
                    nn.Linear(interim, self.hidden_dim, dtype=self.dtype),
                )
        self.embedding_norm = nn.Sequential(
            Transpose(1, 2),
            nn.BatchNorm1d(self.hidden_dim),
            Transpose(1, 2),
        )

    def _build_cnn(self) -> None:
        """Build the CNN stack used after embedding fusion."""
        layers: List[nn.Module] = [Transpose(1, 2)]
        for _ in range(self.cnn_layers):
            layers += [
                nn.Conv1d(
                    self.hidden_dim,
                    self.hidden_dim,
                    kernel_size=2,
                    stride=1,
                    padding=1,
                    dtype=self.dtype,
                ),
                nn.ReLU(),
                MaxPool1D(kernel_size=2, stride=2),
            ]
        layers.append(Transpose(1, 2))
        self.cnn = nn.Sequential(*layers)

    def _build_lstm(self, dropout: float, batch_first: bool) -> None:
        """Build the stacked LSTM."""
        self.lstm = nn.LSTM(
            input_size=self.hidden_dim,
            hidden_size=self.hidden_dim * 8,
            num_layers=2,
            dropout=dropout,
            batch_first=batch_first,
            dtype=self.dtype,
        )

    def _build_head(self) -> None:
        """Build the dense classification head."""
        self.dense = nn.Sequential(
            nn.BatchNorm1d(self.hidden_dim * 8, dtype=self.dtype),
            nn.Linear(self.hidden_dim * 8, self.hidden_dim, dtype=self.dtype),
            nn.ReLU(),
            nn.BatchNorm1d(self.hidden_dim, dtype=self.dtype),
            nn.Linear(self.hidden_dim, self.output_dim, dtype=self.dtype),
        )

    def init_weights(self) -> None:
        """Xavier-normal initialization for matrix-shaped parameters."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_normal_(p)

    # -- inference helpers ----------------------------------------------------

    @torch.no_grad()
    def soft_update(
        self, new_model: "CNNLSTMPredictor", alpha: float = 0.99
    ) -> None:
        """Exponential-moving-average update from another predictor.

        Implements ``theta_self = alpha * theta_self + (1 - alpha) * theta_new``
        in-place, which is the target-network update rule from the paper
        (Appendix D, Eq. 7).

        Args:
            new_model: Source predictor whose weights to blend in.
            alpha: EMA coefficient in ``[0, 1]``. ``alpha=1`` leaves ``self``
                unchanged; ``alpha=0`` fully copies ``new_model`` into ``self``.
        """
        src = new_model.state_dict()
        tgt = self.state_dict()
        for k in tgt:
            tgt[k].copy_(alpha * tgt[k] + (1.0 - alpha) * src[k])

    def pack_sequences(
        self, src: torch.Tensor, mask: torch.Tensor
    ) -> torch.nn.utils.rnn.PackedSequence:
        """Pack a padded batch for efficient LSTM consumption.

        Args:
            src: Embedded sequence tensor ``[batch, seq_len, hidden]``.
            mask: Boolean padding mask ``[batch, seq_len]`` where ``True``
                marks padded positions.

        Returns:
            A ``PackedSequence`` usable by ``self.lstm``.
        """
        lengths = (~mask).sum(-1)
        if lengths.min() == 0:
            mask = mask.clone()
            mask[torch.where(lengths == 0)[0], 0] = False
            lengths = torch.where(lengths == 0, torch.ones_like(lengths), lengths)
        return pack_padded_sequence(
            src, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

    def get_mask_after_conv(self, mask: torch.Tensor) -> torch.Tensor:
        """Propagate a padding mask through the CNN pooling stack.

        Args:
            mask: Boolean mask ``[batch, seq_len]`` aligned with the input.

        Returns:
            Boolean mask aligned with the post-CNN sequence length.
        """
        pooled = mask.unsqueeze(1).float()
        for _ in range(self.cnn_layers):
            pooled = F.avg_pool1d(pooled, kernel_size=2, stride=2)
        return pooled.squeeze(1) == 1

    def standardise_inputs(
        self,
        timepoints: torch.Tensor,
        values: torch.Tensor,
        features: torch.Tensor,
        delta_time: torch.Tensor,
        delta_value: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Apply per-feature z-score standardization.

        For each of ``values``, ``delta_time`` and ``delta_value``, scatter
        the event's value into a feature-indexed long tensor, apply per-
        feature mean/std, then collapse back to a dense per-event tensor
        using ``torch.nansum``. ``timepoints`` use a single global scaler.

        Args:
            timepoints: Event timepoints ``[batch, seq_len]``.
            values: Event values ``[batch, seq_len]``.
            features: Feature ids ``[batch, seq_len]`` with ``-1`` for padding.
            delta_time: Time since previous measurement of the same feature.
            delta_value: Value change since previous measurement of the same
                feature.

        Returns:
            Tuple of standardized tensors in the order
            ``(timepoints, values, delta_time, delta_value, features)``
            where ``features`` has had its ``-1`` sentinels remapped to ``0``.
        """
        safe_time_std = torch.where(
            self.std_timepoints == 0,
            torch.ones_like(self.std_timepoints),
            self.std_timepoints,
        )
        standardised = [(timepoints - self.mean_timepoints) / safe_time_std]

        features = torch.where(features == -1, 0, features)
        long_shape = values.shape[:2] + (self.n_features,)

        base_nan = torch.full(
            long_shape,
            float("nan"),
            dtype=self.dtype,
            device=values.device,
        )

        stats = {
            "values": (self.mean_values, self.std_values),
            "delta_time": (self.mean_delta_time, self.std_delta_time),
            "delta_value": (self.mean_delta_value, self.std_delta_value),
        }
        for name, vector in [
            ("values", values),
            ("delta_time", delta_time),
            ("delta_value", delta_value),
        ]:
            missing = torch.isnan(vector)
            scattered = base_nan.clone()
            scattered.scatter_(-1, features.unsqueeze(-1), vector.unsqueeze(-1))

            mean_v, std_v = stats[name]
            safe_std = torch.where(std_v == 0, torch.ones_like(std_v), std_v)
            scattered = (scattered - mean_v) / safe_std
            collapsed = torch.nansum(scattered, -1).unsqueeze(-1)
            collapsed = torch.where(
                missing.unsqueeze(-1),
                torch.tensor(float("nan"), dtype=self.dtype, device=values.device),
                collapsed,
            )
            standardised.append(collapsed)

        standardised.append(features)
        return tuple(standardised)

    # -- forward --------------------------------------------------------------

    def forward(
        self,
        timepoints: torch.Tensor,
        values: torch.Tensor,
        features: torch.Tensor,
        delta_time: torch.Tensor,
        delta_value: torch.Tensor,
        normalise: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Run the CNN-LSTM forward pass.

        Args:
            timepoints: ``[batch, seq_len]`` hours since intime, NaN for pad.
            values: ``[batch, seq_len]`` measurement values, NaN for pad.
            features: ``[batch, seq_len]`` long tensor of feature ids, -1 for
                pad.
            delta_time: ``[batch, seq_len]`` hours since previous measurement
                of the same feature.
            delta_value: ``[batch, seq_len]`` change in value since previous
                measurement of the same feature.
            normalise: If ``True``, apply ``standardise_inputs`` first.

        Returns:
            Tuple ``(probs, logits)`` each of shape ``[batch, output_dim]``.
            ``probs = sigmoid(logits)``.
        """
        if normalise:
            (
                timepoints,
                values,
                delta_time,
                delta_value,
                features,
            ) = self.standardise_inputs(
                timepoints, values, features, delta_time, delta_value
            )

        timepoints = timepoints.squeeze(-1) if timepoints.ndim == 3 else timepoints
        values = values.squeeze(-1) if values.ndim == 3 else values
        delta_time = delta_time.squeeze(-1) if delta_time.ndim == 3 else delta_time
        delta_value = (
            delta_value.squeeze(-1) if delta_value.ndim == 3 else delta_value
        )

        # Sort descending by timepoint; NaNs (padding) go to the end.
        argsort_idx = torch.argsort(
            torch.where(torch.isnan(timepoints), -torch.inf, timepoints),
            dim=1,
            descending=True,
        )
        timepoints = torch.gather(timepoints, 1, argsort_idx)
        values = torch.gather(values, 1, argsort_idx)
        features = torch.gather(features, 1, argsort_idx)
        delta_time = torch.gather(delta_time, 1, argsort_idx)
        delta_value = torch.gather(delta_value, 1, argsort_idx)

        src_mask = torch.isnan(timepoints)
        timepoints = torch.where(src_mask, 0, timepoints)
        values = torch.where(src_mask, 0, values)
        features = torch.where(src_mask, 0, features)

        dt_mask = torch.isnan(delta_time)
        dv_mask = torch.isnan(delta_value)
        delta_time = torch.where(dt_mask, 0, delta_time)
        delta_value = torch.where(dv_mask, 0, delta_value)

        time_emb = self.embedding_net["time"](timepoints.unsqueeze(-1))
        value_emb = self.embedding_net["value"](values.unsqueeze(-1))
        feature_emb = self.embedding_net["feature"](features)
        dt_emb = self.embedding_net["delta_time"](delta_time.unsqueeze(-1))
        dv_emb = self.embedding_net["delta_value"](delta_value.unsqueeze(-1))

        dt_emb = torch.where(dt_mask.unsqueeze(-1), 0, dt_emb)
        dv_emb = torch.where(dv_mask.unsqueeze(-1), 0, dv_emb)

        embedded = time_emb + value_emb + feature_emb + dt_emb + dv_emb
        embedded = self.embedding_norm(embedded)
        embedded = self.cnn(embedded)

        src_mask = self.get_mask_after_conv(src_mask)

        if self.device_name == "cuda":
            packed = self.pack_sequences(embedded, src_mask)
            embedded = self.lstm(packed)[1][0][-1]
        else:
            embedded = embedded.clone()
            embedded[src_mask] = 0
            embedded = self.lstm(embedded)[1][0][-1]

        logits = self.dense(embedded)
        probs = torch.sigmoid(logits)
        return probs, logits


# -----------------------------------------------------------------------------
# Loss
# -----------------------------------------------------------------------------


class WeightedBCELoss(nn.Module):
    """Thin wrapper around ``BCEWithLogitsLoss`` with optional pos-weighting.

    Args:
        pos_weight: Optional positive-class weight tensor. When ``None``,
            a standard unweighted BCE is used. Only set this for supervised
            training, never for TD training: a weighted loss produces
            incorrect gradients when the target is a continuous bootstrapped
            value rather than a 0/1 label.
    """

    def __init__(self, pos_weight: Optional[torch.Tensor] = None) -> None:
        super().__init__()
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(
        self, logits: torch.Tensor, targets: torch.Tensor
    ) -> torch.Tensor:
        """Compute binary cross-entropy loss from logits.

        Args:
            logits: Raw model output ``[batch, 1]``.
            targets: Target values in ``[0, 1]`` (can be continuous for TD).

        Returns:
            Scalar loss tensor.
        """
        return self.loss_fn(logits, targets.float())


# -----------------------------------------------------------------------------
# PyHealth-facing TD model
# -----------------------------------------------------------------------------


class TDICUMortalityModel(BaseModel):
    """Temporal-difference ICU mortality prediction model.

    Implements the method of Frost et al. (2024), arXiv:2411.04285, for
    real-time mortality prediction in the ICU. The model holds two
    ``CNNLSTMPredictor`` instances:

    * ``online_net``: updated by gradient descent on a TD target.
    * ``target_net``: an EMA-lagged copy of the online net whose predictions
      form part of the TD target.

    At each training step, for a sample transition ``(s_t, s_{t+1})``, the
    training target is:

    * ``target = label`` if ``s_{t+1}`` does not exist (terminal transition);
    * ``target = gamma * target_net(s_{t+1})`` otherwise.

    The online net is regressed against this target with an unweighted binary
    cross-entropy loss. The target net is updated once per optimizer step via
    ``soft_update_target()``.

    The model can also be used in a purely supervised mode
    (``train_td=False``) for baseline comparisons; in this mode the real
    ``label_key`` is used as the target and ``pos_weight`` (if provided) is
    applied.

    Example:
        >>> from pyhealth.datasets import SampleEHRDataset
        >>> from pyhealth.models.td_icu_mortality import TDICUMortalityModel
        >>> dataset = SampleEHRDataset(samples=[], dataset_name="td_icu_demo")
        >>> scaling = build_scaling_dict(...)  # see ``examples/`` directory
        >>> feature_names = [f"feat_{i}" for i in range(128)]
        >>> model = TDICUMortalityModel(
        ...     dataset=dataset,
        ...     feature_keys=["timepoints", "values", "features",
        ...                   "delta_time", "delta_value"],
        ...     label_key="28_day_died",
        ...     n_features=128,
        ...     scaling=scaling,
        ...     features_vocab=feature_names,
        ... )
        >>> out = model(batch, targets=targets, train_td=True)
        >>> out["loss"].backward()
        >>> model.soft_update_target()

    Args:
        dataset: A ``SampleEHRDataset`` instance (can be empty; only used for
            metadata such as ``input_schema`` / ``output_schema``).
        feature_keys: The five keys in the batch dict that hold the event
            tuple components, in the order
            ``[timepoints, values, features, delta_time, delta_value]``.
        label_key: Key in the ``targets`` dict holding the real outcome
            (e.g. ``"28_day_died"``).
        mode: PyHealth task mode. Only ``"binary"`` is currently supported.
        n_features: Size of the feature vocabulary.
        hidden_dim: Hidden size for the encoder and LSTM.
        cnn_layers: Number of CNN pooling blocks.
        dropout: Dropout between LSTM layers.
        output_dim: Output dimensionality (must be 1 for binary mortality).
        scaling: Per-feature mean/std dictionary (see ``CNNLSTMPredictor``).
        features_vocab: Ordered list of feature names keyed by ``scaling``.
        td_alpha: Target-network EMA coefficient. Paper uses 0.99.
        gamma: Discount factor on the bootstrapped next-state value. Default
            1.0 matches the paper.
        pos_weight: Optional positive-class weight for supervised mode only.
        device: Device string passed to the underlying predictors.

    Raises:
        ValueError: If ``mode`` is not ``"binary"``.
    """

    def __init__(
        self,
        dataset: SampleEHRDataset,
        feature_keys: List[str],
        label_key: str,
        mode: str = "binary",
        n_features: int = 128,
        hidden_dim: int = 32,
        cnn_layers: int = 2,
        dropout: float = 0.5,
        output_dim: int = 1,
        scaling: Optional[Mapping[str, Any]] = None,
        features_vocab: Optional[List[str]] = None,
        td_alpha: float = 0.99,
        gamma: float = 1.0,
        pos_weight: Optional[torch.Tensor] = None,
        device: str = "cpu",
    ) -> None:
        if mode != "binary":
            raise ValueError(
                f"TDICUMortalityModel only supports mode='binary', got {mode!r}"
            )
        if scaling is None or features_vocab is None:
            raise ValueError(
                "scaling and features_vocab are required - see the "
                "example script for how to build them from training data."
            )

        # Adapt the SampleEHRDataset for BaseModel's expectations.
        dataset.feature_keys = feature_keys
        dataset.label_key = label_key
        dataset.mode = mode
        if not hasattr(dataset, "input_schema"):
            dataset.input_schema = {k: "sequence" for k in feature_keys}
        if not hasattr(dataset, "output_schema"):
            dataset.output_schema = {label_key: mode}

        super().__init__(dataset)

        self.feature_keys = feature_keys
        self.label_key = label_key
        self.mode = mode
        self.device_name = device
        self.td_alpha = td_alpha
        self.gamma = gamma

        self.online_net = CNNLSTMPredictor(
            n_features=n_features,
            features=features_vocab,
            output_dim=output_dim,
            scaling=scaling,
            cnn_layers=cnn_layers,
            hidden_dim=hidden_dim,
            dropout=dropout,
            device=device,
        )
        self.target_net = CNNLSTMPredictor(
            n_features=n_features,
            features=features_vocab,
            output_dim=output_dim,
            scaling=scaling,
            cnn_layers=cnn_layers,
            hidden_dim=hidden_dim,
            dropout=dropout,
            device=device,
        )
        self.target_net.load_state_dict(deepcopy(self.online_net.state_dict()))

        # pos_weight only ever applied in supervised mode. A continuous TD
        # target combined with pos_weight would silently yield wrong grads.
        self.supervised_loss = WeightedBCELoss(pos_weight=pos_weight)
        self.td_loss = WeightedBCELoss(pos_weight=None)

        self.to(device)

    # -- PyHealth BaseModel abstract methods ---------------------------------

    def prepare_labels(self, labels: torch.Tensor) -> torch.Tensor:
        """Format labels into the BCE-compatible shape.

        Args:
            labels: Tensor of shape ``[batch]`` or ``[batch, 1]``.

        Returns:
            Float tensor of shape ``[batch, 1]``.
        """
        return labels.float().view(-1, 1)

    def get_loss_function(self) -> nn.Module:
        """Return the default (supervised) loss function.

        The TD path uses ``self.td_loss`` internally. ``get_loss_function``
        exists to satisfy PyHealth's ``BaseModel`` contract and returns the
        supervised (optionally pos-weighted) loss used when ``train_td=False``.

        Returns:
            The supervised ``WeightedBCELoss`` instance.
        """
        return self.supervised_loss

    # -- TD-specific helpers --------------------------------------------------

    def predict_current(
        self, batch: Mapping[str, torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Run the online net on the current-state window.

        Args:
            batch: Mapping with the five feature keys listed in
                ``self.feature_keys``.

        Returns:
            Tuple ``(probs, logits)``.
        """
        return self.online_net(
            batch["timepoints"],
            batch["values"],
            batch["features"],
            batch["delta_time"],
            batch["delta_value"],
        )

    @torch.no_grad()
    def predict_next_target(
        self, batch: Mapping[str, torch.Tensor]
    ) -> torch.Tensor:
        """Run the target net on the next-state window.

        The target net is placed in eval mode so BatchNorm stats are frozen
        and LSTM dropout is inactive. The output carries no gradient.

        Args:
            batch: Mapping with the five ``next_*`` keys
                (``next_timepoints``, ``next_values``, ``next_features``,
                ``next_delta_time``, ``next_delta_value``).

        Returns:
            Detached probabilities of shape ``[batch, output_dim]``.
        """
        self.target_net.eval()
        probs, _ = self.target_net(
            batch["next_timepoints"],
            batch["next_values"],
            batch["next_features"],
            batch["next_delta_time"],
            batch["next_delta_value"],
        )
        return probs.detach()

    def compute_td_target(
        self,
        batch: Mapping[str, torch.Tensor],
        targets: Mapping[str, torch.Tensor],
    ) -> torch.Tensor:
        """Compute the TD target for each transition in the batch.

        At terminal transitions (``isterminal > 0.5``) the target is the real
        mortality label; at non-terminal transitions it is
        ``gamma * target_net(next_state)``.

        Args:
            batch: Mapping containing at minimum ``isterminal`` and the five
                ``next_*`` keys.
            targets: Mapping containing ``self.label_key``.

        Returns:
            Detached target tensor of shape ``[batch, output_dim]``.
        """
        next_probs = self.predict_next_target(batch)
        real_reward = targets[self.label_key].float().view_as(next_probs)
        is_terminal = (batch["isterminal"] > 0.5).view_as(next_probs)
        td_target = torch.where(is_terminal, real_reward, self.gamma * next_probs)
        return td_target.detach()

    def soft_update_target(self) -> None:
        """Update the target net with an EMA step from the online net.

        Uses ``self.td_alpha``. Call exactly once after each optimizer step.
        """
        self.target_net.soft_update(self.online_net, alpha=self.td_alpha)

    # -- Forward pass ---------------------------------------------------------

    def forward(
        self,
        batch: Mapping[str, torch.Tensor],
        targets: Optional[Mapping[str, torch.Tensor]] = None,
        train_td: bool = False,
    ) -> Dict[str, Optional[torch.Tensor]]:
        """Run the model.

        Args:
            batch: Mapping with ``self.feature_keys`` and (for TD training)
                also ``isterminal`` and the five ``next_*`` keys.
            targets: Optional mapping with ``self.label_key``. If omitted,
                no loss is computed.
            train_td: Whether to use the temporal-difference loss. When
                ``True``, the TD target is computed via the target net;
                when ``False``, the real label is used directly.

        Returns:
            A dict with keys:

            * ``loss``: scalar loss tensor or ``None`` if ``targets is None``.
            * ``y_prob``: ``[batch, output_dim]`` probabilities from the
              online net.
            * ``y_true``: the supplied label tensor, or ``None``.
            * ``logit``: ``[batch, output_dim]`` raw model output.
        """
        probs, logits = self.predict_current(batch)

        out: Dict[str, Optional[torch.Tensor]] = {
            "loss": None,
            "y_prob": probs,
            "y_true": targets[self.label_key] if targets is not None else None,
            "logit": logits,
        }

        if targets is not None:
            if train_td:
                td_target = self.compute_td_target(batch, targets)
                out["loss"] = self.td_loss(logits, td_target)
            else:
                supervised_target = self.prepare_labels(targets[self.label_key])
                out["loss"] = self.supervised_loss(logits, supervised_target)

        return out

    # -- Monte Carlo dropout for uncertainty quantification ------------------

    @staticmethod
    def _enable_dropout(module: nn.Module) -> None:
        """Put only ``Dropout`` and ``LSTM`` layers into train mode.

        PyTorch's ``LSTM`` applies dropout between stacked layers only when
        the module is in train mode, so we flip both ``nn.Dropout*`` and
        ``nn.LSTM`` modules to train while leaving BatchNorm etc. in eval.

        Args:
            module: Any ``nn.Module`` whose dropout we want to activate.
        """
        for m in module.modules():
            if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d, nn.LSTM)):
                m.train()

    @torch.no_grad()
    def predict_with_confidence(
        self,
        batch: Mapping[str, torch.Tensor],
        n_mc_samples: int = 30,
        high_conf_threshold: float = 0.005,
        low_conf_threshold: float = 0.01,
    ) -> Dict[str, torch.Tensor]:
        """Return mortality predictions with per-patient confidence.

        Runs the online network ``n_mc_samples`` times with dropout active
        (BatchNorm frozen in eval), then reports the MC mean, standard
        deviation, and a ~95% credible interval per sample. Two boolean
        flags classify each prediction as high / low confidence relative
        to user-tunable thresholds on the MC standard deviation.

        This is the paper's base predictor extended with Monte Carlo
        dropout, providing a per-patient uncertainty estimate alongside
        the point mortality probability. Empirically, higher MC standard
        deviation correlates with higher true mortality rate, so the
        confidence score also serves as an auxiliary clinical-triage
        signal.

        Example:
            >>> out = model.predict_with_confidence(batch, n_mc_samples=30)
            >>> mortality = out["mortality_prob"]        # shape [batch]
            >>> ci_lo     = out["ci_95_lower"]           # shape [batch]
            >>> ci_hi     = out["ci_95_upper"]           # shape [batch]
            >>> uncertain = out["is_low_confidence"]     # bool tensor

        Args:
            batch: Mapping with ``self.feature_keys`` (the five event
                tuple components). ``next_*`` keys are not required since
                only the online network is sampled.
            n_mc_samples: Number of stochastic forward passes. More passes
                tighten the uncertainty estimate at linear inference cost.
                30 is a reasonable default; 50-100 for research-grade.
            high_conf_threshold: Predictions with MC standard deviation
                below this value are flagged "high confidence".
            low_conf_threshold: Predictions with MC standard deviation
                above this value are flagged "low confidence".

        Returns:
            Dict with tensor entries all shaped ``[batch]``:

            * ``mortality_prob``: MC-averaged probability.
            * ``confidence_std``: standard deviation across MC samples.
            * ``ci_95_lower``: ``clip(mean - 2*std, 0, 1)``.
            * ``ci_95_upper``: ``clip(mean + 2*std, 0, 1)``.
            * ``is_high_confidence``: bool, std < ``high_conf_threshold``.
            * ``is_low_confidence``: bool, std > ``low_conf_threshold``.
        """
        self.online_net.eval()
        self._enable_dropout(self.online_net)

        probs_mc: List[torch.Tensor] = []
        for _ in range(n_mc_samples):
            probs, _ = self.online_net(
                batch["timepoints"],
                batch["values"],
                batch["features"],
                batch["delta_time"],
                batch["delta_value"],
            )
            probs_mc.append(probs.float().squeeze(-1))

        stacked = torch.stack(probs_mc, dim=1)  # [batch, n_mc_samples]
        mean_prob = stacked.mean(dim=1)
        std_prob = stacked.std(dim=1)

        ci_lower = (mean_prob - 2.0 * std_prob).clamp(0.0, 1.0)
        ci_upper = (mean_prob + 2.0 * std_prob).clamp(0.0, 1.0)

        return {
            "mortality_prob": mean_prob,
            "confidence_std": std_prob,
            "ci_95_lower": ci_lower,
            "ci_95_upper": ci_upper,
            "is_high_confidence": std_prob < high_conf_threshold,
            "is_low_confidence": std_prob > low_conf_threshold,
        }


## 7) Build the Model and Train

In [36]:
def build_model(
    n_features: int,
    feature_names: List[str],
    scaling: Dict[str, Any],
    hidden_dim: int = 32,
    cnn_layers: int = 2,
    dropout: float = 0.5,
    td_alpha: float = 0.99,
    device: str = "cpu",
) -> TDICUMortalityModel:
    """Build a fresh TDICUMortalityModel.

    Args:
        n_features: Feature vocabulary size.
        feature_names: Ordered feature names.
        scaling: Per-feature mean/std dict.
        hidden_dim: Encoder hidden size.
        cnn_layers: Number of CNN blocks.
        dropout: LSTM dropout.
        td_alpha: Target-network EMA coefficient.
        device: Device string.

    Returns:
        Initialized model.
    """
    dataset = SampleEHRDataset(samples=[], dataset_name="td_icu_demo")
    return TDICUMortalityModel(
        dataset=dataset,
        feature_keys=[
            "timepoints", "values", "features",
            "delta_time", "delta_value",
        ],
        label_key="28-day-died",
        mode="binary",
        n_features=n_features,
        hidden_dim=hidden_dim,
        cnn_layers=cnn_layers,
        dropout=dropout,
        scaling=scaling,
        features_vocab=feature_names,
        td_alpha=td_alpha,
        device=device,
    )


def _move(
    batch: Dict[str, torch.Tensor],
    targets: Dict[str, torch.Tensor],
    device: str,
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    """Move batch + targets to ``device`` with non-blocking transfers."""
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    targets = {
        k: v.to(device, non_blocking=True) for k, v in targets.items()
    }
    return batch, targets


In [37]:
def train_one_epoch_real(
    model: TDICUMortalityModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    max_batches: Optional[int] = None,
) -> Dict[str, float]:
    """Train one epoch on a real DataLoader.

    Args:
        model: Model to train.
        loader: DataLoader yielding ``(batch_dict, targets_dict, metas)``.
        optimizer: Optimizer for ``model.online_net`` parameters.
        device: Device string.
        max_batches: Cap on number of batches (for smoke tests).

    Returns:
        Metrics dict with ``loss``, ``auroc``, ``auprc``, ``pos_rate``.
    """
    model.train()
    losses, y_true_all, y_prob_all = [], [], []
    n_batches = (
        len(loader) if max_batches is None else min(len(loader), max_batches)
    )
    pbar = tqdm(enumerate(loader), total=n_batches, desc="train")
    for step, (batch, targets, _) in pbar:
        if max_batches is not None and step >= max_batches:
            break
        batch, targets = _move(batch, targets, device)
        out = model(batch, targets=targets, train_td=True)
        if not torch.isfinite(out["loss"]):
            continue
        optimizer.zero_grad(set_to_none=True)
        out["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.online_net.parameters(), 1.0)
        optimizer.step()
        model.soft_update_target()
        losses.append(out["loss"].item())
        y_true_all.append(out["y_true"].detach().float().cpu().numpy().ravel())
        y_prob_all.append(out["y_prob"].detach().float().cpu().numpy().ravel())

    y_true = np.concatenate(y_true_all) if y_true_all else np.empty(0)
    y_prob = np.concatenate(y_prob_all) if y_prob_all else np.empty(0)
    return _metrics(losses, y_true, y_prob)


@torch.no_grad()
def eval_real(
    model: TDICUMortalityModel,
    loader: DataLoader,
    device: str,
    max_batches: Optional[int] = None,
) -> Dict[str, float]:
    """Evaluate in supervised mode on a real DataLoader.

    Args:
        model: Model to evaluate.
        loader: DataLoader.
        device: Device string.
        max_batches: Cap on batches.

    Returns:
        Metrics dict.
    """
    model.eval()
    losses, y_true_all, y_prob_all = [], [], []
    n_batches = (
        len(loader) if max_batches is None else min(len(loader), max_batches)
    )
    pbar = tqdm(enumerate(loader), total=n_batches, desc="val")
    for step, (batch, targets, _) in pbar:
        if max_batches is not None and step >= max_batches:
            break
        batch, targets = _move(batch, targets, device)
        out = model(batch, targets=targets, train_td=False)
        losses.append(out["loss"].item())
        y_true_all.append(out["y_true"].float().cpu().numpy().ravel())
        y_prob_all.append(out["y_prob"].float().cpu().numpy().ravel())
    y_true = np.concatenate(y_true_all) if y_true_all else np.empty(0)
    y_prob = np.concatenate(y_prob_all) if y_prob_all else np.empty(0)
    return _metrics(losses, y_true, y_prob)


def _metrics(
    losses: List[float],
    y_true: np.ndarray,
    y_prob: np.ndarray,
) -> Dict[str, float]:
    """Summarize a training / eval pass."""
    out: Dict[str, float] = {
        "loss": float(np.mean(losses)) if losses else float("nan"),
        "n_samples": int(len(y_true)),
    }
    if len(y_true) > 0 and len(np.unique(y_true)) > 1:
        out["auroc"] = float(roc_auc_score(y_true, y_prob))
        out["auprc"] = float(average_precision_score(y_true, y_prob))
    else:
        out["auroc"] = float("nan")
        out["auprc"] = float("nan")
    out["pos_rate"] = float(y_true.mean()) if len(y_true) else float("nan")
    return out


In [51]:
def _load_feature_names_and_scaling(
    data_root: Path,
) -> Tuple[List[str], Dict[str, Any]]:
    """Load feature names and scaling dict from standard filenames."""
    feature_file = data_root / "features.txt"
    scaling_file = data_root / "scaling.pt"
    train_h5 = data_root / "h5train_fixed.hdf5"

    if not feature_file.exists():
        raise FileNotFoundError(
            f"Expected {feature_file}. Save the ordered feature list one "
            "name per line."
        )
    feature_names = [
        ln.strip()
        for ln in feature_file.read_text().splitlines()
        if ln.strip()
    ]

    if scaling_file.exists():
        scaling = torch.load(
            scaling_file, map_location="cpu", weights_only=False,
        )
        print(f"loaded scaling from {scaling_file}")
    elif train_h5.exists():
        print(f"computing scaling from {train_h5}...")
        scaling = compute_scaling_from_h5(train_h5, feature_names)
        torch.save(scaling, scaling_file)
        print(f"saved {scaling_file}")
    else:
        raise FileNotFoundError(
            f"Neither {scaling_file} nor {train_h5} found. Run --mode build "
            "first or provide both."
        )
    return feature_names, scaling

def _save_checkpoint(
    path: Path,
    model: TDICUMortalityModel,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val_auroc: float,
) -> None:
    """Save a checkpoint dict (model + optimizer + metadata).

    The format mirrors the training notebook shipped with the paper so
    that ``run_evaluate`` can consume checkpoints from either source.

    Args:
        path: Destination ``.pt`` file.
        model: The TDICUMortalityModel to snapshot.
        optimizer: Optimizer whose state is saved alongside weights.
        epoch: Current epoch number (for resume / provenance).
        best_val_auroc: Best validation AUROC seen so far.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_auroc": best_val_auroc,
            "label_key": model.label_key,
        },
        path,
    )

def _load_checkpoint_into(
    path: Path,
    model: TDICUMortalityModel,
    optimizer: Optional[torch.optim.Optimizer] = None,
    device: str = "cpu",
) -> Dict[str, Any]:
    """Load a checkpoint into ``model`` (and optionally ``optimizer``).

    Handles both wrapped dicts (``{"model_state_dict": ..., ...}``) and
    raw ``state_dict`` files.

    Args:
        path: Source ``.pt`` file.
        model: Model to load weights into (mutated in place).
        optimizer: Optional optimizer to restore alongside weights.
        device: ``map_location`` target for ``torch.load``.

    Returns:
        The full checkpoint dict if the file was wrapped, else
        ``{"model_state_dict": state_dict}``.
    """
    ckpt = torch.load(path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
        if optimizer is not None and "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        return ckpt
    # Raw state_dict - backward compatibility
    model.load_state_dict(ckpt)
    return {"model_state_dict": ckpt}

def run_real_train(
    data_root: Path,
    build_first: bool,
    batch_size: int = 64,
    num_workers: int = 4,
    max_train_batches: Optional[int] = None,
    max_val_batches: Optional[int] = None,
    epochs: int = 1,
    device: str = "cpu",
) -> None:
    """Train on real MIMIC-IV data (optionally build HDF5 first).

    Args:
        data_root: Directory containing parquets and/or pre-built HDF5s.
        build_first: If ``True``, build HDF5 from parquets before training.
        batch_size: DataLoader batch size.
        num_workers: DataLoader workers.
        max_train_batches: Cap on train batches per epoch.
        max_val_batches: Cap on val batches.
        epochs: Number of epochs.
        device: Device string.
    """
    print(f"=== Mode: {'build' if build_first else 'load'} ===")
    data_root = Path(data_root)
    ckpt_dir = Path(checkpoint_dir) if checkpoint_dir else data_root / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    last_ckpt = ckpt_dir / "last.pt"
    best_ckpt = ckpt_dir / "best.pt"
    print(f"checkpoint dir: {ckpt_dir}")

    if build_first:
        print("Building HDF5 files from parquets...")
        for split in ["val", "train", "test"]:
            build_hdf5_from_parquets(
                parquet_dir=data_root, out_dir=data_root, split=split,
            )

    feature_names, scaling = _load_feature_names_and_scaling(data_root)
    n_features = len(feature_names)
    print(f"n_features={n_features}")

    label_keys = [
        "1-day-died", "3-day-died", "7-day-died",
        "14-day-died", "28-day-died",
    ]

    train_ds = H5SequenceDataset(
        data_root / "h5train_fixed.hdf5", label_keys=label_keys,
    )
    val_ds = H5SequenceDataset(
        data_root / "h5val_fixed.hdf5", label_keys=label_keys,
    )
    test_ds = H5SequenceDataset(
        data_root / "h5test_fixed.hdf5", label_keys=label_keys,
    )
    print(
        f"train samples: {len(train_ds):,}, "
        f"val samples: {len(val_ds):,}, "
        f"test samples: {len(test_ds):,}"
    )

    loader_kwargs: Dict[str, Any] = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
        collate_fn=collate_td,
        persistent_workers=(num_workers > 0),
    )
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    model = build_model(
        n_features=n_features,
        feature_names=feature_names,
        scaling=scaling,
        hidden_dim=32,
        cnn_layers=2,
        dropout=0.5,
        td_alpha=0.99,
        device=device,
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    lr = 1.0 / math.sqrt(n_params)
    wd = 1 / (lr * len(train_ds))

    optimizer = torch.optim.AdamW(
        model.online_net.parameters(), lr=lr, weight_decay=wd,
    )
    best_val_auroc = -1.0


    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch_real(
            model, train_loader, optimizer, device, max_train_batches,
    )
    val_metrics = eval_real(model, val_loader, device, max_val_batches)
    print(f"[epoch {epoch}] train {train_metrics} | val {val_metrics}")

    # Always save last.pt after each epoch
    _save_checkpoint(
            last_ckpt, model, optimizer, epoch, best_val_auroc,
    )
    # Save best.pt when val AUROC improves
    val_auroc = val_metrics.get("auroc", float("nan"))
    if np.isfinite(val_auroc) and val_auroc > best_val_auroc:
      best_val_auroc = val_auroc
      _save_checkpoint(
                best_ckpt, model, optimizer, epoch, best_val_auroc,
      )
      print(
                f"  [best] val_auroc={best_val_auroc:.4f} "
                f"saved to {best_ckpt}"
      )


## 8) Execute the Model Training

In [ ]:
data_root = out_root
run_real_train(
            data_root=data_root,
            build_first=True,
            batch_size=128,
            num_workers=6,
            max_train_batches=None,
            max_val_batches=None,
            epochs=1,
            device=device,
)

## 9) Evaluation functions

In [39]:
@torch.no_grad()
def evaluate_all_horizons(
    model: TDICUMortalityModel,
    loader: DataLoader,
    label_keys: List[str],
    device: str,
    max_batches: Optional[int] = None,
) -> Dict[str, Dict[str, float]]:
    """Evaluate the model across all mortality horizons on a test loader.

    The TD model is trained on a single horizon (e.g. 28-day mortality)
    but its probability output is used to score every horizon. Matches
    the paper's Table 2 layout.

    Args:
        model: Trained ``TDICUMortalityModel``.
        loader: Test ``DataLoader`` (must yield all ``label_keys`` in the
            targets dict).
        label_keys: List of horizon names (e.g. ``["1-day-died", ...]``).
        device: Device string.
        max_batches: Optional cap for smoke tests.

    Returns:
        Mapping from horizon name to a dict with ``auroc``, ``auprc``,
        ``pos_rate``, ``n``.
    """
    model.eval()
    y_prob_all: List[np.ndarray] = []
    y_true_by_key: Dict[str, List[np.ndarray]] = {k: [] for k in label_keys}

    n_batches = (
        len(loader) if max_batches is None else min(len(loader), max_batches)
    )
    pbar = tqdm(enumerate(loader), total=n_batches, desc="test")
    for step, (batch, targets, _) in pbar:
        if max_batches is not None and step >= max_batches:
            break
        batch, targets = _move(batch, targets, device)
        probs, _ = model.online_net(
            batch["timepoints"], batch["values"], batch["features"],
            batch["delta_time"], batch["delta_value"],
        )
        y_prob_all.append(probs.float().cpu().numpy().ravel())
        for k in label_keys:
            y_true_by_key[k].append(targets[k].float().cpu().numpy().ravel())

    y_prob = np.concatenate(y_prob_all)
    results: Dict[str, Dict[str, float]] = {}
    for k in label_keys:
        y_true = np.concatenate(y_true_by_key[k])
        if len(np.unique(y_true)) > 1:
            results[k] = {
                "auroc": float(roc_auc_score(y_true, y_prob)),
                "auprc": float(average_precision_score(y_true, y_prob)),
            }
        else:
            results[k] = {
                "auroc": float("nan"),
                "auprc": float("nan"),
            }
        results[k]["pos_rate"] = float(y_true.mean())
        results[k]["n"] = int(len(y_true))
    return results


def print_horizon_table(
    results: Dict[str, Dict[str, float]],
    trained_on: str = "28-day-died",
) -> None:
    """Pretty-print the horizon results table (paper-style).

    Args:
        results: Output of ``evaluate_all_horizons``.
        trained_on: Label key the model was actually trained on.
    """
    bar = "=" * 60
    print(f"\n{bar}")
    print(f"TEST SET RESULTS (trained on {trained_on} via TD)")
    print(bar)
    print(
        f"{'horizon':<12}  {'AUROC':>7}  {'AUPRC':>7}  "
        f"{'pos_rate':>8}  {'n':>10}"
    )
    for k, r in results.items():
        print(
            f"{k:<12}  {r['auroc']:>7.4f}  {r['auprc']:>7.4f}  "
            f"{r['pos_rate']:>8.4f}  {r['n']:>10,}"
        )


## 10) EXTENSION - MC DROPOUT

In [40]:
@torch.no_grad()
def mc_dropout_predict_test(
    model: TDICUMortalityModel,
    loader: DataLoader,
    device: str,
    n_mc_samples: int = 30,
    label_key: str = "28-day-died",
    max_batches: Optional[int] = None,
) -> Dict[str, np.ndarray]:
    """Run MC dropout inference across a full test loader.

    For each sample, runs ``n_mc_samples`` stochastic forward passes of
    the online network (with dropout active, BatchNorm frozen) and
    collects the full distribution of predictions.

    Args:
        model: Trained ``TDICUMortalityModel``.
        loader: Test ``DataLoader``.
        device: Device string.
        n_mc_samples: Number of stochastic forward passes per input.
        label_key: Which label to compare against.
        max_batches: Optional cap for smoke tests.

    Returns:
        Dict of numpy arrays, each length ``N`` (total test samples):

        * ``y_true``: ground-truth labels
        * ``mean_prob``: MC-averaged probabilities
        * ``std_prob``: MC standard deviation (epistemic uncertainty)
        * ``ci_lower``, ``ci_upper``: 95% credible interval bounds
    """
    model.online_net.eval()
    # activate only dropout, keep BatchNorm frozen
    for m in model.online_net.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d, nn.LSTM)):
            m.train()

    y_true_all: List[np.ndarray] = []
    mean_all: List[np.ndarray] = []
    std_all: List[np.ndarray] = []

    n_batches = (
        len(loader) if max_batches is None else min(len(loader), max_batches)
    )
    pbar = tqdm(enumerate(loader), total=n_batches, desc="mc-dropout")
    for step, (batch, targets, _) in pbar:
        if max_batches is not None and step >= max_batches:
            break
        batch, targets = _move(batch, targets, device)
        probs_mc = []
        for _ in range(n_mc_samples):
            probs, _ = model.online_net(
                batch["timepoints"], batch["values"], batch["features"],
                batch["delta_time"], batch["delta_value"],
            )
            probs_mc.append(probs.float().cpu().numpy().ravel())
        stacked = np.stack(probs_mc, axis=1)  # [batch, n_mc]
        mean_all.append(stacked.mean(axis=1))
        std_all.append(stacked.std(axis=1))
        y_true_all.append(targets[label_key].float().cpu().numpy().ravel())

    mean_prob = np.concatenate(mean_all)
    std_prob = np.concatenate(std_all)
    y_true = np.concatenate(y_true_all)
    return {
        "y_true": y_true,
        "mean_prob": mean_prob,
        "std_prob": std_prob,
        "ci_lower": np.clip(mean_prob - 2.0 * std_prob, 0.0, 1.0),
        "ci_upper": np.clip(mean_prob + 2.0 * std_prob, 0.0, 1.0),
    }


def print_mc_dropout_analysis(mc: Dict[str, np.ndarray]) -> None:
    """Print the paper-extension's MC dropout analysis tables.

    Reports:
      - Deterministic vs MC-averaged AUROC / AUPRC / Brier
      - Uncertainty distribution summary
      - Stratified-by-uncertainty-quintile table (the paper-extension's
        key finding: higher uncertainty correlates with higher mortality)

    Args:
        mc: Output of ``mc_dropout_predict_test``.
    """
    y_true = mc["y_true"]
    mean_prob = mc["mean_prob"]
    std_prob = mc["std_prob"]

    bar = "=" * 60

    # MC-averaged metrics
    print(f"\n{bar}")
    print("MC-DROPOUT TEST RESULTS")
    print(bar)
    if len(np.unique(y_true)) > 1:
        auroc = float(roc_auc_score(y_true, mean_prob))
        auprc = float(average_precision_score(y_true, mean_prob))
        brier = float(np.mean((mean_prob - y_true) ** 2))
        print(f"  AUROC (MC-averaged): {auroc:.4f}")
        print(f"  AUPRC (MC-averaged): {auprc:.4f}")
        print(f"  Brier (MC-averaged): {brier:.4f}")

    # Uncertainty distribution
    print(f"\nUncertainty distribution (std across MC samples):")
    print(f"  mean:   {std_prob.mean():.4f}   median: {np.median(std_prob):.4f}")
    print(
        f"  p5:     {np.percentile(std_prob, 5):.4f}   "
        f"p95:    {np.percentile(std_prob, 95):.4f}"
    )
    print(f"  max:    {std_prob.max():.4f}")

    # Stratified by uncertainty quintile
    print(f"\n{bar}")
    print("STRATIFIED BY UNCERTAINTY QUINTILE")
    print(bar)
    print(
        f"  {'bin':<5}  {'std range':<17}  {'n':>6}  "
        f"{'pos_rate':>8}  {'auroc':>7}  {'brier':>7}"
    )
    quintile_bounds = np.quantile(std_prob, np.linspace(0, 1, 6))
    for i in range(5):
        lo, hi = quintile_bounds[i], quintile_bounds[i + 1]
        mask = (std_prob >= lo) & (std_prob <= hi) if i == 4 \
            else (std_prob >= lo) & (std_prob < hi)
        if mask.sum() < 10:
            continue
        y_b = y_true[mask]
        p_b = mean_prob[mask]
        if len(np.unique(y_b)) > 1:
            auroc_b = float(roc_auc_score(y_b, p_b))
        else:
            auroc_b = float("nan")
        brier_b = float(np.mean((p_b - y_b) ** 2))
        print(
            f"  {i + 1:<5}  [{lo:.3f}, {hi:.3f}]  {mask.sum():>6,}  "
            f"{y_b.mean():>8.3f}  {auroc_b:>7.4f}  {brier_b:>7.4f}"
        )


def print_clinical_triage_demo(
    mc: Dict[str, np.ndarray],
    high_conf_threshold: float = 0.005,
    low_conf_threshold: float = 0.01,
) -> None:
    """Show the clinical triage interpretation of MC dropout confidence.

    Splits predictions into three categories and reports the actual
    mortality rate in each. This is the paper-extension's key clinical
    finding: the confidence score doubles as an auxiliary triage signal.

    Args:
        mc: Output of ``mc_dropout_predict_test``.
        high_conf_threshold: MC std below which a prediction is "confident".
        low_conf_threshold: MC std above which a prediction is "uncertain".
    """
    y_true = mc["y_true"]
    mean_prob = mc["mean_prob"]
    std_prob = mc["std_prob"]

    is_high_conf = std_prob < high_conf_threshold
    is_low_conf = std_prob > low_conf_threshold

    auto_alert = is_high_conf & (mean_prob > 0.5)
    review = is_low_conf
    standard = is_high_conf & (mean_prob < 0.2)

    bar = "=" * 60
    print(f"\n{bar}")
    print("CLINICAL TRIAGE DEMO (paper-extension claim)")
    print(bar)
    print(
        f"  auto_alert   (high conf, high risk):   "
        f"n={int(auto_alert.sum()):>6,}   "
        f"mortality={y_true[auto_alert].mean() * 100:>5.1f}%"
        if auto_alert.sum() > 0 else
        f"  auto_alert   (high conf, high risk):   n=0"
    )
    print(
        f"  review       (low conf):               "
        f"n={int(review.sum()):>6,}   "
        f"mortality={y_true[review].mean() * 100:>5.1f}%"
        if review.sum() > 0 else
        f"  review       (low conf):               n=0"
    )
    print(
        f"  standard     (high conf, low risk):    "
        f"n={int(standard.sum()):>6,}   "
        f"mortality={y_true[standard].mean() * 100:>5.1f}%"
        if standard.sum() > 0 else
        f"  standard     (high conf, low risk):    n=0"
    )
    print(
        f"  overall:                               "
        f"n={len(y_true):>6,}   "
        f"mortality={y_true.mean() * 100:>5.1f}%"
    )

In [41]:
def run_evaluate(
    data_root: Path,
    checkpoint_path: Path,
    batch_size: int = 128,
    num_workers: int = 0,
    max_test_batches: Optional[int] = None,
    run_mc_dropout: bool = False,
    n_mc_samples: int = 30,
    device: str = "cpu",
) -> None:
    """Evaluate a trained checkpoint on the test set. No training.

    Loads a saved ``state_dict`` into a fresh model, builds only the test
    DataLoader (no train/val loaders), runs ``evaluate_all_horizons`` to
    produce the paper's per-horizon AUROC/AUPRC table, and optionally
    runs the MC dropout analysis.

    Use this when you already have a trained checkpoint and just want to
    see test metrics, without paying the full training cost again.

    Args:
        data_root: Directory with pre-built HDF5s (``h5test_fixed.hdf5``,
            ``features.txt``, and either ``scaling.pt`` or ``h5train_fixed.hdf5``).
        checkpoint_path: Path to a ``.pt`` file containing either a raw
            ``state_dict`` or a dict with a ``model_state_dict`` key
            (matches the format produced by the training loop).
        batch_size: DataLoader batch size.
        num_workers: DataLoader workers.
        max_test_batches: Cap on test batches (None = full test set).
        run_mc_dropout: Whether to run MC dropout analysis.
        n_mc_samples: Number of stochastic forward passes per sample.
        device: Device string.
    """
    print("=== Mode: evaluate ===")
    data_root = Path(data_root)
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found at {checkpoint_path}"
        )

    feature_names, scaling = _load_feature_names_and_scaling(data_root)
    n_features = len(feature_names)
    print(f"n_features={n_features}")

    label_keys = [
        "1-day-died", "3-day-died", "7-day-died",
        "14-day-died", "28-day-died",
    ]

    test_h5 = data_root / "h5test_fixed.hdf5"
    if not test_h5.exists():
        raise FileNotFoundError(
            f"Test HDF5 not found at {test_h5}. "
            "Run --mode build first to produce it."
        )
    test_ds = H5SequenceDataset(test_h5, label_keys=label_keys)
    print(f"test samples: {len(test_ds):,}")

    loader_kwargs: Dict[str, Any] = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
        collate_fn=collate_td,
        persistent_workers=(num_workers > 0),
    )
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    # Build a fresh model with the same architecture as training
    model = build_model(
        n_features=n_features,
        feature_names=feature_names,
        scaling=scaling,
        hidden_dim=32,
        cnn_layers=2,
        dropout=0.5,
        td_alpha=0.99,
        device=device,
    )

    # Load checkpoint using the shared helper
    print(f"loading checkpoint from {checkpoint_path}")
    ckpt = _load_checkpoint_into(
        checkpoint_path, model, optimizer=None, device=device,
    )
    print("checkpoint loaded")
    if "best_val_auroc" in ckpt:
        print(f"  checkpoint best_val_auroc: {ckpt['best_val_auroc']:.4f}")
    if "epoch" in ckpt:
        print(f"  checkpoint epoch:          {ckpt['epoch']}")

    # ----- Test-set evaluation across all horizons -----
    print("\nEvaluating on test set across all mortality horizons...")
    test_results = evaluate_all_horizons(
        model=model,
        loader=test_loader,
        label_keys=label_keys,
        device=device,
        max_batches=max_test_batches,
    )
    print_horizon_table(test_results, trained_on=model.label_key)

    # ----- MC dropout analysis on test set -----
    if run_mc_dropout:
        print(
            f"\nRunning MC dropout inference "
            f"({n_mc_samples} samples/input)..."
        )
        mc = mc_dropout_predict_test(
            model=model,
            loader=test_loader,
            device=device,
            n_mc_samples=n_mc_samples,
            label_key=model.label_key,
            max_batches=max_test_batches,
        )
        print_mc_dropout_analysis(mc)
        print_clinical_triage_demo(mc)

## 11) Run Evaluation - Option to run with/without the MC Dropout Extension

In [48]:
checkpoint_path = checkpoint_dir / 'best.pt'
run_evaluate(
    data_root=data_root,
    checkpoint_path=checkpoint_path ,
    batch_size=128,
    num_workers=6,
    max_test_batches=None,
    run_mc_dropout=False,
    n_mc_samples=30,
    device=device,
)

=== Mode: evaluate ===
loaded scaling from /content/data/mimic/scaling.pt
n_features=108
test samples: 295,085
loading checkpoint from /content/checkpoints/best.pt
checkpoint loaded
  checkpoint best_val_auroc: 0.8110
  checkpoint epoch:          5

Evaluating on test set across all mortality horizons...


test:   0%|          | 0/2306 [00:00<?, ?it/s]


TEST SET RESULTS (trained on 28-day-died via TD)
horizon         AUROC    AUPRC  pos_rate           n
1-day-died     0.8275   0.1156    0.0156     295,085
3-day-died     0.8303   0.2281    0.0364     295,085
7-day-died     0.8058   0.3250    0.0760     295,085
14-day-died    0.7893   0.4244    0.1326     295,085
28-day-died    0.7652   0.4748    0.1857     295,085


## 12) ABLATION - CHANGE ALPHA PARAMETERS

In [ ]:
def run_alpha_ablation(
    mode: str = "load",
    data_root: Optional[Path] = None,
    alpha_values: Tuple[float, ...] = (0.90, 0.95, 0.99, 0.995, 0.999),
    n_train_steps: int = 100,
    max_train_batches: int = 200,
    max_val_batches: int = 50,
    device: str = "cpu",
) -> None:
    """Sweep target-network EMA alpha.

    Args:
        mode: ``"load"``.
        data_root: Required when ``mode="load"``.
        alpha_values: Alphas to try.
        n_train_steps: Steps per alpha (synthetic mode).
        max_train_batches: Batches per alpha (load mode).
        max_val_batches: Val batches per alpha (load mode).
        device: Device string.
    """
    print("=== alpha ablation ===")

    if mode == "load":
        if data_root is None:
            raise ValueError("--data-root is required when mode=load")
        feature_names, scaling = _load_feature_names_and_scaling(data_root)
        n_features = len(feature_names)
        label_keys = [
            "1-day-died", "3-day-died", "7-day-died",
            "14-day-died", "28-day-died",
        ]
        train_ds = H5SequenceDataset(
            data_root / "h5train_fixed.hdf5", label_keys=label_keys,
        )
        val_ds = H5SequenceDataset(
            data_root / "h5val_fixed.hdf5", label_keys=label_keys,
        )
        loader_kwargs: Dict[str, Any] = dict(
            batch_size=128,
            num_workers=4,
            pin_memory=(device == "cuda"),
            collate_fn=collate_td,
            persistent_workers=True,
        )
        train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
        val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    else:
        raise ValueError(f"Unknown ablation mode {mode!r}")

    print(
        f"{'alpha':>6}  {'loss_mean':>10}  {'loss_std':>10}  "
        f"{'val_auroc':>10}  {'val_loss':>10}"
    )
    all_results = []
    alpha_val = []
    for alpha in alpha_values:
        torch.manual_seed(42)
        np.random.seed(42)
        model = build_model(
            n_features=n_features,
            feature_names=feature_names,
            scaling=scaling,
            hidden_dim=32,
            td_alpha=alpha,
            device=device,
        )

        optimizer = torch.optim.AdamW(
                model.online_net.parameters(), lr=1e-3,
        )
        result = train_one_epoch_real(
                model, train_loader, optimizer, device,
                max_batches=max_train_batches,
        )
        #all_results.append(result)
        val = eval_real(
                model, val_loader, device, max_batches=max_val_batches,
        )
        all_results.append(val)
        alpha_val.append(alpha)
        losses = [val["loss"], val["loss"]]  # placeholder for stats
        tail = (
            np.array(losses[-50:]) if len(losses) > 1 else np.array(losses)
        )
        print(
            f"{alpha:>6.3f}  {tail.mean():>10.4f}  {tail.std():>10.4f}  "
            f"{val.get('auroc', float('nan')):>10.4f}  "
            f"{val['loss']:>10.4f}"
        )
    # ----------------------------- final summary ---------------------------------
    i = 0
    print("\n" + "=" * 60)
    print("ALPHA ABLATION SUMMARY")
    print("=" * 60)
    print(f"{'alpha':>6}  {'val_auroc':>10}  {'val_auprc':>10}  "
    f"{'loss':>10}  {'pos_rate':>10} ")
    for r in all_results:
        print(
        f"{alpha_val[i]:>6.3f}  {r['auroc']:>10.4f}  {r['auprc']:>10.4f}  "
        f"{r['loss']:>10.4f}  {r['pos_rate']:>10.4f}  "
        )
        i=i+1


In [ ]:
run_alpha_ablation(
    mode="load",
    data_root=data_root,
    max_train_batches=4,
    max_val_batches=2,
    device=device,
)